In [1]:
# ============================================================
# EDUSHIELD
# DATA AUDIT & CLEANING NOTEBOOK
# ============================================================
#
# Project:
# EduShield - Academic Integrity & AI Authorship Analytics
#
# Purpose:
# This notebook audits the raw synthetic datasets generated
# for the EduShield project and prepares them for the cleaning
# and validation stage.
#
# Audit areas:
# - Dataset structure
# - Data types
# - Missing values
# - Duplicate records
# - Categorical inconsistencies
# - Numerical anomalies
# - Datetime consistency
# - Primary-key integrity
# - Foreign-key integrity
# - Cross-table business rules
#
# IMPORTANT:
# The raw datasets are treated as source data.
# No cleaning or modification will be performed during the
# initial audit phase.
# ============================================================


import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("EduShield Data Audit started.")

EduShield Data Audit started.


In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

current_path = Path.cwd()

possible_roots = [
    current_path,
    *current_path.parents
]

PROJECT_ROOT = None

for path in possible_roots:

    if (
        (path / "data" / "raw").exists()
    ):
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate EduShield/data/raw folder. "
        "Check the notebook location."
    )


RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR = PROJECT_ROOT / "audit_reports"
GENERATED_CLEAN_DIR = PROJECT_ROOT / "data" / "generated_clean"


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

print("\nAudit reports directory:")
print(AUDIT_DIR)

print("\nGenerated clean directory:")
print(GENERATED_CLEAN_DIR)

Project root:
c:\Users\adity\Documents\EduShield

Raw data directory:
c:\Users\adity\Documents\EduShield\data\raw

Processed data directory:
c:\Users\adity\Documents\EduShield\data\processed

Audit reports directory:
c:\Users\adity\Documents\EduShield\audit_reports

Generated clean directory:
c:\Users\adity\Documents\EduShield\data\generated_clean


In [3]:
# ============================================================
# RAW DATASET INVENTORY
# ============================================================

TABLE_FILES = {
    "academic_terms": "academic_terms.csv",
    "departments": "departments.csv",
    "majors": "majors.csv",
    "instructors": "instructors.csv",
    "students": "students.csv",
    "courses": "courses.csv",
    "course_offerings": "course_offerings.csv",
    "assignments": "assignments.csv",
    "submissions": "submissions.csv",
    "detector_tools": "detector_tools.csv",
    "ai_detector_results": "ai_detector_results.csv",
    "student_behavior_events": "student_behavior_events.csv",
    "integrity_cases": "integrity_cases.csv",
    "student_integrity_history": "student_integrity_history.csv"
}


print(
    f"Expected raw datasets: {len(TABLE_FILES)}"
)

for table_name, filename in TABLE_FILES.items():
    print(
        f"{table_name:<32} → {filename}"
    )

Expected raw datasets: 14
academic_terms                   → academic_terms.csv
departments                      → departments.csv
majors                           → majors.csv
instructors                      → instructors.csv
students                         → students.csv
courses                          → courses.csv
course_offerings                 → course_offerings.csv
assignments                      → assignments.csv
submissions                      → submissions.csv
detector_tools                   → detector_tools.csv
ai_detector_results              → ai_detector_results.csv
student_behavior_events          → student_behavior_events.csv
integrity_cases                  → integrity_cases.csv
student_integrity_history        → student_integrity_history.csv


In [4]:
# ============================================================
# FILE EXISTENCE CHECK
# ============================================================

file_check_rows = []

for table_name, filename in TABLE_FILES.items():

    file_path = RAW_DIR / filename

    file_check_rows.append({

        "table_name": table_name,

        "filename": filename,

        "exists": file_path.exists(),

        "file_size_kb": (
            round(
                file_path.stat().st_size / 1024,
                2
            )
            if file_path.exists()
            else None
        )
    })


file_check = pd.DataFrame(
    file_check_rows
)

display(file_check)

missing_files = file_check[
    ~file_check["exists"]
]

if len(missing_files) > 0:

    raise FileNotFoundError(
        "One or more expected raw files are missing."
    )

print(
    f"\nAll {len(TABLE_FILES)} raw dataset files found."
)

,table_name,filename,exists,file_size_kb
0,academic_terms,academic_terms.csv,True,0.58
1,departments,departments.csv,True,0.28
2,majors,majors.csv,True,0.72
3,instructors,instructors.csv,True,5.51
4,students,students.csv,True,148.07
5,courses,courses.csv,True,6.92
6,course_offerings,course_offerings.csv,True,15.11
7,assignments,assignments.csv,True,166.07
8,submissions,submissions.csv,True,3508.04
9,detector_tools,detector_tools.csv,True,0.56



All 14 raw dataset files found.


In [5]:
# ============================================================
# LOAD RAW DATA
# ============================================================
#
# IMPORTANT:
# The synthetic data legitimately uses the category "None"
# in fields such as scholarship_status and sanction_level.
#
# pandas normally interprets "None" as NaN when reading CSVs.
# We therefore explicitly preserve "None" as a valid value
# while still treating empty CSV fields as missing.
# ============================================================

data = {}

for table_name, filename in TABLE_FILES.items():

    file_path = RAW_DIR / filename

    data[table_name] = pd.read_csv(
        file_path,
        low_memory=False,
        keep_default_na=False,
        na_values=[""]
    )


print("All raw datasets loaded successfully.\n")

for table_name, df in data.items():

    print(
        f"{table_name:<32}"
        f"{len(df):>9,} rows × "
        f"{len(df.columns):>3} columns"
    )

All raw datasets loaded successfully.

academic_terms                          7 rows ×   6 columns
departments                            11 rows ×   3 columns
majors                                 25 rows ×   3 columns
instructors                           101 rows ×   7 columns
students                            1,523 rows ×  13 columns
courses                                91 rows ×   8 columns
course_offerings                      263 rows ×   8 columns
assignments                           812 rows ×  20 columns
submissions                        15,234 rows ×  32 columns
detector_tools                          6 rows ×   8 columns
ai_detector_results                75,944 rows ×  11 columns
student_behavior_events            10,177 rows ×  10 columns
integrity_cases                     1,397 rows ×  23 columns
student_integrity_history           7,586 rows ×  13 columns


In [6]:
# ============================================================
# DATAFRAME REFERENCES
# ============================================================

terms = data["academic_terms"]
departments = data["departments"]
majors = data["majors"]
instructors = data["instructors"]
students = data["students"]
courses = data["courses"]
course_offerings = data["course_offerings"]
assignments = data["assignments"]
submissions = data["submissions"]
detector_tools = data["detector_tools"]
ai_detector_results = data["ai_detector_results"]
behavior_events = data["student_behavior_events"]
integrity_cases = data["integrity_cases"]
integrity_history = data["student_integrity_history"]

In [7]:
# ============================================================
# DATASET-LEVEL AUDIT
# ============================================================

overview_rows = []

for table_name, df in data.items():

    overview_rows.append({

        "table_name":
            table_name,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "missing_cells":
            int(
                df.isna()
                .sum()
                .sum()
            ),

        "duplicate_rows":
            int(
                df.duplicated()
                .sum()
            ),

        "memory_mb":
            round(
                df.memory_usage(
                    deep=True
                ).sum()
                / (1024 ** 2),
                2
            )
    })


dataset_overview = pd.DataFrame(
    overview_rows
)

dataset_overview

,table_name,rows,columns,missing_cells,duplicate_rows,memory_mb
0,academic_terms,7,6,0,1,0.00
1,departments,11,3,0,1,0.00
2,majors,25,3,0,1,0.00
3,instructors,101,7,6,1,0.01
4,students,1523,13,150,15,0.26
5,courses,91,8,3,1,0.01
6,course_offerings,263,8,10,2,0.03
7,assignments,812,20,64,8,0.25
8,submissions,15234,32,3350,151,4.86
9,detector_tools,6,8,1,1,0.00


In [8]:
# ============================================================
# COLUMN-LEVEL AUDIT
# ============================================================

column_audit_rows = []

for table_name, df in data.items():

    for column in df.columns:

        series = df[column]

        column_audit_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "dtype":
                str(series.dtype),

            "non_null_count":
                int(
                    series.notna().sum()
                ),

            "missing_count":
                int(
                    series.isna().sum()
                ),

            "missing_percentage":
                round(
                    series.isna().mean() * 100,
                    2
                ),

            "unique_values":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),

            "duplicate_values":
                int(
                    series.duplicated().sum()
                )
        })


column_audit = pd.DataFrame(
    column_audit_rows
)

column_audit

,table_name,column_name,dtype,non_null_count,missing_count,missing_percentage,unique_values,duplicate_values
0,academic_terms,term_id,str,7,0,0.00,5,2
1,academic_terms,term_name,str,7,0,0.00,6,1
2,academic_terms,academic_year,str,7,0,0.00,2,5
3,academic_terms,term_type,str,7,0,0.00,4,3
4,academic_terms,start_date,str,7,0,0.00,6,1
...,...,...,...,...,...,...,...,...
160,student_integrity_history,prior_ai_flags,int64,7586,0,0.00,12,7574
161,student_integrity_history,prior_plagiarism_flags,int64,7586,0,0.00,6,7580
162,student_integrity_history,prior_sanction_points,int64,7586,0,0.00,18,7568
163,student_integrity_history,recent_integrity_events,int64,7586,0,0.00,10,7576


In [9]:
# ============================================================
# MISSING VALUE AUDIT
# ============================================================

missing_rows = []

for table_name, df in data.items():

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        if missing_count > 0:

            missing_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "missing_count":
                    missing_count,

                "missing_percentage":
                    round(
                        missing_count
                        / len(df)
                        * 100,
                        2
                    )
            })


missing_audit = pd.DataFrame(
    missing_rows
)

missing_audit = (
    missing_audit
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print(
    f"Columns with missing values: "
    f"{len(missing_audit)}"
)

missing_audit

Columns with missing values: 49


,table_name,column_name,missing_count,missing_percentage
0,integrity_cases,appeal_date,1253,89.69
1,integrity_cases,sanction_type,1009,72.23
2,student_behavior_events,event_value,1822,17.90
3,detector_tools,model_version,1,16.67
4,integrity_cases,investigation_duration_days,128,9.16
5,integrity_cases,case_closed_date,128,9.16
6,ai_detector_results,writing_style_score,2854,3.76
7,ai_detector_results,detector_perplexity_score,2703,3.56
8,ai_detector_results,text_length,2501,3.29
9,ai_detector_results,confidence_score,2274,2.99


In [10]:
# ============================================================
# DUPLICATE AUDIT
# ============================================================

duplicate_audit_rows = []

for table_name, df in data.items():

    duplicate_count = int(
        df.duplicated()
        .sum()
    )

    duplicate_audit_rows.append({

        "table_name":
            table_name,

        "total_rows":
            len(df),

        "duplicate_rows":
            duplicate_count,

        "duplicate_percentage":
            round(
                duplicate_count
                / len(df)
                * 100,
                3
            )
    })


duplicate_audit = pd.DataFrame(
    duplicate_audit_rows
)

duplicate_audit

,table_name,total_rows,duplicate_rows,duplicate_percentage
0,academic_terms,7,1,14.286
1,departments,11,1,9.091
2,majors,25,1,4.000
3,instructors,101,1,0.990
4,students,1523,15,0.985
5,courses,91,1,1.099
6,course_offerings,263,2,0.760
7,assignments,812,8,0.985
8,submissions,15234,151,0.991
9,detector_tools,6,1,16.667


In [11]:
# ============================================================
# NUMERIC AUDIT
# ============================================================

numeric_audit_rows = []

for table_name, df in data.items():

    for column in df.columns:

        numeric_series = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        valid_numeric = (
            numeric_series.notna()
        )

        if not valid_numeric.any():
            continue

        numeric_audit_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "numeric_values":
                int(valid_numeric.sum()),

            "minimum":
                numeric_series.min(),

            "maximum":
                numeric_series.max(),

            "mean":
                numeric_series.mean(),

            "median":
                numeric_series.median()
        })


numeric_audit = pd.DataFrame(
    numeric_audit_rows
)

numeric_audit

,table_name,column_name,numeric_values,minimum,maximum,mean,median
0,instructors,years_experience,99,1.0,29.0,10.545455,10.00
1,instructors,teaching_load,101,0,9,2.603960,2.00
2,students,gpa,1493,-0.4,4.75,3.059518,3.06
3,students,expected_graduation_year,1523,2018,2045,2027.512804,2027.00
4,students,attendance_rate,1493,-8.0,107.0,85.609223,86.08
...,...,...,...,...,...,...,...
71,student_integrity_history,prior_suspicious_flags,7586,-3,50,2.817031,2.00
72,student_integrity_history,prior_ai_flags,7586,-2,50,1.333245,1.00
73,student_integrity_history,prior_plagiarism_flags,7586,-2,50,0.210651,0.00
74,student_integrity_history,prior_sanction_points,7586,-5,100,0.619035,0.00


In [12]:
# ============================================================
# DATETIME AUDIT
# ============================================================

DATE_COLUMNS = {

    "academic_terms": [
        "start_date",
        "end_date"
    ],

    "students": [
        "enrollment_date"
    ],

    "assignments": [
        "assignment_open_at",
        "submission_deadline"
    ],

    "submissions": [
        "submitted_at"
    ],

    "ai_detector_results": [
        "test_timestamp"
    ],

    "student_behavior_events": [
        "event_date"
    ],

    "integrity_cases": [
        "incident_date",
        "case_open_date",
        "appeal_date",
        "case_closed_date"
    ],

    "student_integrity_history": [
        "snapshot_at"
    ]
}


date_audit_rows = []

for table_name, columns in DATE_COLUMNS.items():

    df = data[table_name]

    for column in columns:

        raw_series = df[column]

        parsed = pd.to_datetime(
            raw_series,
            errors="coerce"
        )

        original_missing = (
            raw_series.isna()
        )

        invalid_count = int(
            (
                parsed.isna()
                &
                ~original_missing
            ).sum()
        )

        date_audit_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "valid_dates":
                int(
                    parsed.notna().sum()
                ),

            "invalid_date_values":
                invalid_count,

            "missing_dates":
                int(
                    original_missing.sum()
                ),

            "minimum_date":
                parsed.min(),

            "maximum_date":
                parsed.max()
        })


date_audit = pd.DataFrame(
    date_audit_rows
)

date_audit

,table_name,column_name,valid_dates,invalid_date_values,missing_dates,minimum_date,maximum_date
0,academic_terms,start_date,6,1,0,2025-08-18 00:00:00.000000000,2027-01-11 00:00:00.000000000
1,academic_terms,end_date,6,1,0,2025-12-19 00:00:00.000000000,2027-05-14 00:00:00.000000000
2,students,enrollment_date,1491,32,0,2022-08-01 00:00:00.000000000,2025-10-28 00:00:00.000000000
3,assignments,assignment_open_at,798,14,0,2025-08-28 18:00:00.000000000,2027-03-13 17:00:00.000000000
4,assignments,submission_deadline,793,19,0,2025-09-08 03:00:00.000000000,2027-04-26 05:00:00.000000000
5,submissions,submitted_at,14854,380,0,2025-09-05 03:13:54.202978990,2027-04-27 23:33:15.834142649
6,ai_detector_results,test_timestamp,74049,1895,0,2025-09-05 03:19:53.865797283,2027-04-27 23:43:08.125081970
7,student_behavior_events,event_date,9922,255,0,2025-09-05 17:27:50.191150200,2027-04-28 12:21:36.652275350
8,integrity_cases,incident_date,1370,27,0,2025-09-07 07:24:46.100426338,2027-04-27 17:49:03.654412671
9,integrity_cases,case_open_date,1370,27,0,2025-09-08 13:14:21.081603156,2027-04-28 05:33:44.460356530


In [13]:
# ============================================================
# BOOLEAN REPRESENTATION AUDIT
# ============================================================

BOOLEAN_COLUMNS = {

    "assignments": [
        "requires_citations",
        "research_required",
        "oral_defense_required",
        "collaboration_allowed"
    ],

    "submissions": [
        "late_submission",
        "is_ai_generated"
    ],

    "ai_detector_results": [
        "detected_as_ai"
    ],

    "student_behavior_events": [
        "resolved"
    ],

    "integrity_cases": [
        "plagiarism_flag",
        "ai_detector_flag",
        "network_flag",
        "faculty_report",
        "appealed"
    ],

    "students": []
}


for table_name, columns in BOOLEAN_COLUMNS.items():

    df = data[table_name]

    for column in columns:

        print(
            "\n" + "=" * 60
        )

        print(
            f"{table_name}.{column}"
        )

        print(
            "=" * 60
        )

        print(
            df[column]
            .value_counts(
                dropna=False
            )
            .to_string()
        )


assignments.requires_citations
requires_citations
True     641
False    153
1          6
YES        4
TRUE       3
FALSE      2
yes        2
NO         1

assignments.research_required
research_required
True     625
False    170
1          4
TRUE       4
YES        3
yes        2
no         2
FALSE      1
NO         1

assignments.oral_defense_required
oral_defense_required
False    641
True     154
0          5
NO         3
no         3
YES        2
1          2
FALSE      1
yes        1

assignments.collaboration_allowed
collaboration_allowed
False    561
True     235
no         4
TRUE       3
1          3
yes        2
NO         2
YES        1
0          1

submissions.late_submission
late_submission
False    13070
True      1916
0           58
FALSE       56
NO          52
no          46
yes         13
TRUE         9
1            8
YES          6

submissions.is_ai_generated
is_ai_generated
False    9874
True     5112
FALSE      49
no         43
0          41
NO         31
yes    

In [14]:
# ============================================================
# CATEGORICAL VALUE AUDIT
# ============================================================

CATEGORICAL_COLUMNS = {

    "students": [
        "year_level",
        "academic_standing",
        "scholarship_status",
        "active_status"
    ],

    "courses": [
        "course_level",
        "difficulty_level"
    ],

    "course_offerings": [
        "section_code",
        "delivery_mode"
    ],

    "assignments": [
        "assignment_type",
        "difficulty_level",
        "allowed_ai_usage",
        "design_type"
    ],

    "submissions": [
        "submission_status",
        "ai_generation_source"
    ],

    "detector_tools": [
        "tool_name",
        "active_status"
    ],

    "ai_detector_results": [
        "processing_status"
    ],

    "student_behavior_events": [
        "event_type",
        "severity",
        "description_code"
    ],

    "integrity_cases": [
        "case_type",
        "trigger_source",
        "initial_suspicion_level",
        "student_response",
        "verdict",
        "sanction_level",
        "sanction_type"
    ],

    "student_integrity_history": [
        "integrity_status"
    ]
}


for table_name, columns in CATEGORICAL_COLUMNS.items():

    df = data[table_name]

    print(
        "\n\n" + "#" * 75
    )

    print(
        table_name.upper()
    )

    print(
        "#" * 75
    )

    for column in columns:

        print(
            f"\n--- {column} ---"
        )

        print(
            df[column]
            .value_counts(
                dropna=False
            )
            .to_string()
        )



###########################################################################
STUDENTS
###########################################################################

--- year_level ---
year_level
Junior         425
Sophomore      369
Freshman       357
Senior         335
 Junior          9
 Sophomore       7
 Freshman        4
sophomore        4
 Senior          3
junior           2
freshman         2
senior           2
SENIOR           1
SOPHOMORE        1
FRESHMAN         1
JUNIOR           1

--- academic_standing ---
academic_standing
Good                  1055
Honors                 192
Academic Warning       191
NaN                     30
Probation               21
 Good                   17
good                     7
GOOD                     3
 Academic Warning        2
 Honors                  2
honors                   1
academic warning         1
 Probation               1

--- scholarship_status ---
scholarship_status
None         852
Partial      425
Full         177
NaN     

In [15]:
# ============================================================
# SAVE INITIAL AUDIT REPORTS
# ============================================================

dataset_overview.to_csv(
    AUDIT_DIR / "01_dataset_overview.csv",
    index=False
)

column_audit.to_csv(
    AUDIT_DIR / "02_column_audit.csv",
    index=False
)

missing_audit.to_csv(
    AUDIT_DIR / "03_missing_value_audit.csv",
    index=False
)

duplicate_audit.to_csv(
    AUDIT_DIR / "04_duplicate_audit.csv",
    index=False
)

numeric_audit.to_csv(
    AUDIT_DIR / "05_numeric_audit.csv",
    index=False
)

date_audit.to_csv(
    AUDIT_DIR / "06_datetime_audit.csv",
    index=False
)

print(
    "Initial audit reports saved successfully."
)

Initial audit reports saved successfully.


In [16]:
# ============================================================
# PRIMARY KEY AUDIT
# ============================================================

PRIMARY_KEYS = {

    "academic_terms": [
        "term_id"
    ],

    "departments": [
        "department_id"
    ],

    "majors": [
        "major_id"
    ],

    "instructors": [
        "instructor_id"
    ],

    "students": [
        "student_id"
    ],

    "courses": [
        "course_id"
    ],

    "course_offerings": [
        "offering_id"
    ],

    "assignments": [
        "assignment_id"
    ],

    "submissions": [
        "submission_id"
    ],

    "detector_tools": [
        "tool_id"
    ],

    "ai_detector_results": [
        "detector_result_id"
    ],

    "student_behavior_events": [
        "event_id"
    ],

    "integrity_cases": [
        "case_id"
    ],

    "student_integrity_history": [
        "history_id"
    ]
}


pk_audit_rows = []

for table_name, pk_columns in PRIMARY_KEYS.items():

    df = data[table_name]

    for column in pk_columns:

        null_count = int(
            df[column].isna().sum()
        )

        duplicate_count = int(
            df[column].duplicated(
                keep=False
            ).sum()
        )

        pk_audit_rows.append({

            "table_name":
                table_name,

            "primary_key":
                column,

            "null_keys":
                null_count,

            "duplicate_key_values":
                duplicate_count,

            "unique_key_values":
                int(
                    df[column]
                    .nunique(
                        dropna=True
                    )
                )
        })


pk_audit = pd.DataFrame(
    pk_audit_rows
)

pk_audit

,table_name,primary_key,null_keys,duplicate_key_values,unique_key_values
0,academic_terms,term_id,0,4,5
1,departments,department_id,0,2,10
2,majors,major_id,0,2,24
3,instructors,instructor_id,0,2,100
4,students,student_id,0,46,1500
5,courses,course_id,0,2,90
6,course_offerings,offering_id,0,4,261
7,assignments,assignment_id,0,24,800
8,submissions,submission_id,0,464,15000
9,detector_tools,tool_id,0,2,5


In [17]:
# ============================================================
# CLEANING PHASE
# ============================================================
#
# The raw datasets loaded above are preserved unchanged.
# From this point onward, all transformations are performed
# on independent working copies.
#
# Cleaning objectives:
# - Remove exact duplicate records
# - Normalize identifiers and text formatting
# - Standardize categorical values
# - Standardize boolean representations
# - Parse and validate datetime fields
# - Detect and repair invalid numerical values
# - Handle missing values according to business meaning
# - Validate primary keys and foreign keys
# - Validate cross-table temporal relationships
#
# ============================================================


clean_data = {
    table_name: df.copy(deep=True)
    for table_name, df in data.items()
}

print(
    f"Created clean working copies for "
    f"{len(clean_data)} tables."
)

Created clean working copies for 14 tables.


In [18]:
# ============================================================
# CLEANING LOG
# ============================================================

cleaning_log = []


def log_cleaning(
    table,
    column,
    issue,
    action,
    reason
):
    cleaning_log.append({
        "table_name": table,
        "column_name": column,
        "issue": issue,
        "action_taken": action,
        "reason": reason
    })


print("Cleaning log initialized.")

Cleaning log initialized.


In [19]:
# ============================================================
# REMOVE EXACT DUPLICATES
# ============================================================

duplicate_summary = []


for table_name, df in clean_data.items():

    before = len(df)

    duplicate_count = int(
        df.duplicated()
        .sum()
    )

    if duplicate_count > 0:

        clean_data[table_name] = (
            df.drop_duplicates(
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )

        after = len(
            clean_data[table_name]
        )

        log_cleaning(
            table_name,
            "*",
            "Exact duplicate rows",
            f"Removed {duplicate_count} duplicate rows",
            "Exact duplicate records do not represent additional observations."
        )

    else:

        after = before

    duplicate_summary.append({
        "table_name": table_name,
        "rows_before": before,
        "duplicate_rows_removed": (
            before - after
        ),
        "rows_after": after
    })


duplicate_summary = pd.DataFrame(
    duplicate_summary
)

duplicate_summary

,table_name,rows_before,duplicate_rows_removed,rows_after
0,academic_terms,7,1,6
1,departments,11,1,10
2,majors,25,1,24
3,instructors,101,1,100
4,students,1523,15,1508
5,courses,91,1,90
6,course_offerings,263,2,261
7,assignments,812,8,804
8,submissions,15234,151,15083
9,detector_tools,6,1,5


In [20]:
# ============================================================
# PRIMARY KEY RECHECK AFTER DUPLICATE REMOVAL
# ============================================================

pk_recheck_rows = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    df = clean_data[
        table_name
    ]

    for column in pk_columns:

        null_keys = int(
            df[column]
            .isna()
            .sum()
        )

        duplicated_keys = int(
            df[column]
            .duplicated(
                keep=False
            )
            .sum()
        )

        pk_recheck_rows.append({

            "table_name":
                table_name,

            "primary_key":
                column,

            "null_keys":
                null_keys,

            "duplicate_key_rows":
                duplicated_keys,

            "unique_values":
                int(
                    df[column]
                    .nunique(
                        dropna=True
                    )
                )
        })


pk_recheck = pd.DataFrame(
    pk_recheck_rows
)

pk_recheck

,table_name,primary_key,null_keys,duplicate_key_rows,unique_values
0,academic_terms,term_id,0,2,5
1,departments,department_id,0,0,10
2,majors,major_id,0,0,24
3,instructors,instructor_id,0,0,100
4,students,student_id,0,16,1500
5,courses,course_id,0,0,90
6,course_offerings,offering_id,0,0,261
7,assignments,assignment_id,0,8,800
8,submissions,submission_id,0,166,15000
9,detector_tools,tool_id,0,0,5


In [21]:
# ============================================================
# STANDARDIZE WHITESPACE
# ============================================================

whitespace_changes = []


for table_name, df in clean_data.items():

    for column in df.select_dtypes(
        include=["object", "string"]
    ).columns:

        before = df[column].copy()

        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

        changed = (
            before.fillna("<NA>")
            !=
            df[column].fillna("<NA>")
        ).sum()

        if changed > 0:

            whitespace_changes.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "values_changed":
                    int(changed)
            })

            log_cleaning(
                table_name,
                column,
                "Leading/trailing whitespace",
                "Trimmed whitespace",
                "Standardizes identifiers and categorical text."
            )


whitespace_audit = pd.DataFrame(
    whitespace_changes
)

print(
    "Whitespace normalization completed."
)

whitespace_audit.head(30)

Whitespace normalization completed.


,table_name,column_name,values_changed
0,academic_terms,term_name,1
1,academic_terms,term_type,1
2,departments,department_name,1
3,departments,department_code,1
4,majors,major_name,1
5,majors,department_id,1
6,instructors,instructor_name,2
7,instructors,department_id,2
8,instructors,academic_rank,1
9,students,first_name,22


In [22]:
# ============================================================
# NORMALIZE IDENTIFIERS
# ============================================================

IDENTIFIER_COLUMNS = {

    "academic_terms": [
        "term_id"
    ],

    "departments": [
        "department_id"
    ],

    "majors": [
        "major_id",
        "department_id"
    ],

    "instructors": [
        "instructor_id",
        "department_id"
    ],

    "students": [
        "student_id",
        "major_id"
    ],

    "courses": [
        "course_id",
        "department_id"
    ],

    "course_offerings": [
        "offering_id",
        "course_id",
        "term_id",
        "instructor_id"
    ],

    "assignments": [
        "assignment_id",
        "offering_id"
    ],

    "submissions": [
        "submission_id",
        "student_id",
        "assignment_id"
    ],

    "detector_tools": [
        "tool_id"
    ],

    "ai_detector_results": [
        "submission_id",
        "tool_id"
    ],

    "student_behavior_events": [
        "student_id",
        "course_id",
        "related_submission_id"
    ],

    "integrity_cases": [
        "case_id",
        "student_id",
        "submission_id",
        "assignment_id"
    ],

    "student_integrity_history": [
        "student_id"
    ]
}


identifier_changes = []


for table_name, columns in IDENTIFIER_COLUMNS.items():

    df = clean_data[
        table_name
    ]

    for column in columns:

        if column not in df.columns:
            continue

        before = (
            df[column]
            .astype("string")
        )

        after = (
            before
            .str.strip()
            .str.upper()
        )

        changed = (
            before.fillna("<NA>")
            != after.fillna("<NA>")
        ).sum()

        df[column] = after

        if changed > 0:

            identifier_changes.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "values_normalized":
                    int(changed)
            })

            log_cleaning(
                table_name,
                column,
                "Identifier formatting inconsistency",
                "Trimmed and converted identifiers to uppercase",
                "Foreign keys require consistent identifier representation."
            )


identifier_audit = pd.DataFrame(
    identifier_changes
)

print(
    "Identifier normalization completed."
)

identifier_audit

Identifier normalization completed.


""


In [23]:
# ============================================================
# BOOLEAN STANDARDIZATION
# ============================================================

BOOLEAN_MAPPING = {

    "true": True,
    "false": False,

    "yes": True,
    "no": False,

    "1": True,
    "0": False,

    "y": True,
    "n": False,

    "t": True,
    "f": False
}


def normalize_boolean(value):

    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    text = (
        str(value)
        .strip()
        .lower()
    )

    return BOOLEAN_MAPPING.get(
        text,
        pd.NA
    )


boolean_changes = []


for table_name, columns in BOOLEAN_COLUMNS.items():

    df = clean_data[
        table_name
    ]

    for column in columns:

        if column not in df.columns:
            continue

        before = df[column].copy()

        df[column] = (
            df[column]
            .apply(normalize_boolean)
            .astype("boolean")
        )

        changed = (
            before.astype("string")
            .fillna("<NA>")
            !=
            df[column]
            .astype("string")
            .fillna("<NA>")
        ).sum()

        if changed > 0:

            boolean_changes.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "values_standardized":
                    int(changed)
            })

            log_cleaning(
                table_name,
                column,
                "Inconsistent boolean representations",
                "Converted TRUE/FALSE/YES/NO/1/0 variants to Boolean",
                "Boolean fields require a single consistent representation."
            )


boolean_audit = pd.DataFrame(
    boolean_changes
)

print(
    "Boolean standardization completed."
)

boolean_audit.head(30)

Boolean standardization completed.


,table_name,column_name,values_standardized
0,assignments,requires_citations,18
1,assignments,research_required,17
2,assignments,oral_defense_required,17
3,assignments,collaboration_allowed,16
4,submissions,late_submission,247
5,submissions,is_ai_generated,248
6,ai_detector_results,detected_as_ai,1208
7,student_behavior_events,resolved,208
8,integrity_cases,plagiarism_flag,26
9,integrity_cases,ai_detector_flag,26


In [24]:
# ============================================================
# CATEGORICAL STANDARDIZATION MAPS
# ============================================================
#
# All mappings use lowercase, trimmed source values as keys.
# The corresponding values are the final standardized labels.
# ============================================================

CATEGORY_MAPS = {

    # --------------------------------------------------------
    # STUDENTS
    # --------------------------------------------------------

    "students": {

        "year_level": {
            "freshman": "Freshman",
            "sophomore": "Sophomore",
            "junior": "Junior",
            "senior": "Senior"
        },

        "academic_standing": {
            "honors": "Honors",
            "good": "Good",
            "probation": "Probation",
            "academic warning": "Academic Warning"
        },

        "scholarship_status": {
            "none": "None",
            "partial": "Partial",
            "full": "Full"
        },

        "active_status": {
            "active": "Active",
            "graduated": "Graduated",
            "withdrawn": "Withdrawn"
        }
    },


    # --------------------------------------------------------
    # INSTRUCTORS
    # --------------------------------------------------------

    "instructors": {

        "academic_rank": {
            "lecturer": "Lecturer",
            "assistant professor":
                "Assistant Professor",
            "associate professor":
                "Associate Professor",
            "professor": "Professor"
        },

        "ai_policy_adoption": {
            "low": "Low",
            "medium": "Medium",
            "high": "High"
        }
    },


    # --------------------------------------------------------
    # COURSES
    # --------------------------------------------------------

    "courses": {

        "course_level": {
            "introductory": "Introductory",
            "intermediate": "Intermediate",
            "advanced": "Advanced"
        },

        "difficulty_level": {
            "easy": "Easy",
            "moderate": "Moderate",
            "hard": "Hard"
        }
    },


    # --------------------------------------------------------
    # COURSE OFFERINGS
    # --------------------------------------------------------

    "course_offerings": {

        "delivery_mode": {
            "in-person": "In-Person",
            "online": "Online",
            "hybrid": "Hybrid"
        }
    },


    # --------------------------------------------------------
    # ASSIGNMENTS
    # --------------------------------------------------------

    "assignments": {

        "assignment_type": {
            "essay": "Essay",
            "research paper":
                "Research Paper",
            "lab report":
                "Lab Report",
            "case study":
                "Case Study",
            "literature review":
                "Literature Review",
            "reflective report":
                "Reflective Report"
        },

        "difficulty_level": {
            "easy": "Easy",
            "moderate": "Moderate",
            "hard": "Hard"
        },

        "allowed_ai_usage": {
            "prohibited": "Prohibited",
            "limited": "Limited",
            "disclosure required":
                "Disclosure Required",
            "allowed": "Allowed"
        },

        "design_type": {
            "traditional":
                "Traditional",
            "process-oriented":
                "Process-Oriented",
            "oral defense":
                "Oral Defense",
            "project-based":
                "Project-Based",
            "reflective":
                "Reflective",
            "peer-review":
                "Peer-Review",
            "multi-stage":
                "Multi-Stage",
            "scaffolded":
                "Scaffolded"
        }
    },


    # --------------------------------------------------------
    # SUBMISSIONS
    # --------------------------------------------------------

    "submissions": {

        "submission_status": {
            "submitted":
                "Submitted",
            "late":
                "Late",
            "resubmitted":
                "Resubmitted",
            "withdrawn":
                "Withdrawn"
        },

        "ai_generation_source": {
            "human":
                "Human",
            "chatgpt":
                "ChatGPT",
            "claude":
                "Claude",
            "gemini":
                "Gemini",
            "other ai":
                "Other AI",
            "mixed":
                "Mixed"
        }
    },


    # --------------------------------------------------------
    # DETECTOR TOOLS
    # --------------------------------------------------------

    "detector_tools": {

        "tool_name": {
            "gptzero":
                "GPTZero",
            "copyleaks":
                "Copyleaks",
            "turnitin":
                "Turnitin",
            "originality.ai":
                "Originality.ai",
            "winston ai":
                "Winston AI"
        }
    },


    # --------------------------------------------------------
    # AI DETECTOR RESULTS
    # --------------------------------------------------------

    "ai_detector_results": {

        "processing_status": {
            "success":
                "Success",
            "partial":
                "Partial",
            "failed":
                "Failed"
        }
    },


    # --------------------------------------------------------
    # STUDENT BEHAVIOR EVENTS
    # --------------------------------------------------------

    "student_behavior_events": {

        "event_type": {

            "ai detector flag":
                "AI Detector Flag",

            "similarity flag":
                "Similarity Flag",

            "unusual collaboration":
                "Unusual Collaboration",

            "rapid submission":
                "Rapid Submission",

            "plagiarism flag":
                "Plagiarism Flag",

            "excessive copy-paste":
                "Excessive Copy-Paste",

            "repeated revision anomaly":
                "Repeated Revision Anomaly",

            "peer report":
                "Peer Report",

            "unauthorized file sharing":
                "Unauthorized File Sharing"
        },

        "severity": {
            "low": "Low",
            "medium": "Medium",
            "high": "High"
        }
    },


    # --------------------------------------------------------
    # INTEGRITY CASES
    # --------------------------------------------------------

    "integrity_cases": {

        "case_type": {

            "ai misuse":
                "AI Misuse",

            "plagiarism":
                "Plagiarism",

            "academic misconduct":
                "Academic Misconduct",

            "unauthorized collaboration":
                "Unauthorized Collaboration",

            "unauthorized file sharing":
                "Unauthorized File Sharing",

            "other integrity concern":
                "Other Integrity Concern"
        },

        "trigger_source": {

            "ai detector":
                "AI Detector",

            "faculty report":
                "Faculty Report",

            "similarity check":
                "Similarity Check",

            "automated monitoring":
                "Automated Monitoring",

            "peer report":
                "Peer Report",

            "plagiarism screening":
                "Plagiarism Screening"
        },

        "initial_suspicion_level": {
            "low":
                "Low",
            "medium":
                "Medium",
            "high":
                "High"
        },

        "student_response": {

            "admitted":
                "Admitted",

            "denied":
                "Denied",

            "partial admission":
                "Partial Admission",

            "no response":
                "No Response"
        },

        "verdict": {

            "guilty":
                "Guilty",

            "not guilty":
                "Not Guilty",

            "dismissed":
                "Dismissed",

            "insufficient evidence":
                "Insufficient Evidence",

            "withdrawn":
                "Withdrawn",

            "pending":
                "Pending",

            "appealed":
                "Appealed"
        },

        "sanction_level": {

            "none":
                "None",

            "warning":
                "Warning",

            "moderate":
                "Moderate",

            "severe":
                "Severe"
        }
    },


    # --------------------------------------------------------
    # STUDENT INTEGRITY HISTORY
    # --------------------------------------------------------

    "student_integrity_history": {

        "integrity_status": {

            "high":
                "High",

            "moderate":
                "Moderate",

            "low":
                "Low"
        }
    }
}

In [25]:
# ============================================================
# APPLY CATEGORICAL STANDARDIZATION
# ============================================================

category_audit_rows = []


for table_name, column_maps in CATEGORY_MAPS.items():

    df = clean_data[table_name]

    for column, mapping in column_maps.items():

        if column not in df.columns:
            continue

        before = (
            df[column]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        after = before.map(mapping)

        after = after.astype("string")

        changed = (
            before.fillna("<NA>")
            !=
            after.fillna("<NA>")
        ).sum()

        df[column] = after

        if changed > 0:

            category_audit_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "values_standardized":
                    int(changed)
            })

            log_cleaning(
                table_name,
                column,
                "Categorical capitalization/format inconsistency",
                "Mapped values to controlled category labels",
                "Ensures consistent categorical analysis and modeling."
            )


category_audit = pd.DataFrame(
    category_audit_rows
)

print(
    "Categorical standardization completed."
)

category_audit

Categorical standardization completed.


,table_name,column_name,values_standardized
0,students,year_level,1508
1,students,academic_standing,1478
2,students,scholarship_status,1478
3,students,active_status,1508
4,instructors,academic_rank,98
5,instructors,ai_policy_adoption,98
6,courses,course_level,89
7,courses,difficulty_level,89
8,course_offerings,delivery_mode,256
9,assignments,assignment_type,804


In [26]:
# ============================================================
# POST-STANDARDIZATION CATEGORY CHECK
# ============================================================

for table_name, columns in CATEGORICAL_COLUMNS.items():

    df = clean_data[
        table_name
    ]

    print(
        "\n" + "=" * 65
    )

    print(
        table_name.upper()
    )

    print(
        "=" * 65
    )

    for column in columns:

        if column not in df.columns:
            continue

        print(
            f"\n{column}:"
        )

        print(
            df[column]
            .value_counts(
                dropna=False
            )
            .to_string()
        )


STUDENTS

year_level:
year_level
Junior       430
Sophomore    377
Freshman     361
Senior       340

academic_standing:
academic_standing
Good                1071
Honors               194
Academic Warning     191
<NA>                  30
Probation             22

scholarship_status:
scholarship_status
None       869
Partial    430
Full       179
<NA>        30

active_status:
active_status
Active       1327
Graduated     152
Withdrawn      29

COURSES

course_level:
course_level
Intermediate    39
Introductory    33
Advanced        17
<NA>             1

difficulty_level:
difficulty_level
Moderate    47
Hard        23
Easy        19
<NA>         1

COURSE_OFFERINGS

section_code:
section_code
C    73
A    67
D    61
B    60

delivery_mode:
delivery_mode
In-Person    172
Hybrid        58
Online        26
<NA>           5

ASSIGNMENTS

assignment_type:
assignment_type
Research Paper       216
Essay                199
Lab Report           137
Case Study           111
Literature Review  

In [27]:
# ============================================================
# DATETIME STANDARDIZATION
# ============================================================
#
# Raw data may contain:
# - ISO format
# - slash-separated dates
# - day-first dates
# - different time representations
#
# We normalize all recognized values into pandas datetime.
# Invalid/unparseable values become NaT and are handled
# separately during the missing/invalid-value stage.
# ============================================================


datetime_cleaning_summary = []


for table_name, columns in DATE_COLUMNS.items():

    df = clean_data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        before = df[column].copy()

        # Parse mixed datetime representations.
        parsed = pd.to_datetime(
            before,
            format="mixed",
            errors="coerce"
        )

        # Count values that were present but could not be parsed.
        invalid_count = int(
            (
                parsed.isna()
                &
                before.notna()
                &
                (
                    before.astype("string").str.strip() != ""
                )
            ).sum()
        )

        df[column] = parsed

        datetime_cleaning_summary.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "values_before":
                len(before),

            "valid_datetime_values":
                int(parsed.notna().sum()),

            "invalid_datetime_values":
                invalid_count,

            "missing_values":
                int(parsed.isna().sum())
        })

        log_cleaning(
            table_name,
            column,
            "Mixed datetime representations",
            "Parsed mixed date/time formats into datetime64",
            "Creates a consistent temporal data type for analysis and cross-table validation."
        )


datetime_cleaning_summary = pd.DataFrame(
    datetime_cleaning_summary
)

datetime_cleaning_summary

,table_name,column_name,values_before,valid_datetime_values,invalid_datetime_values,missing_values
0,academic_terms,start_date,6,6,0,0
1,academic_terms,end_date,6,6,0,0
2,students,enrollment_date,1508,1508,0,0
3,assignments,assignment_open_at,804,804,0,0
4,assignments,submission_deadline,804,804,0,0
5,submissions,submitted_at,15083,15083,0,0
6,ai_detector_results,test_timestamp,75192,75192,0,0
7,student_behavior_events,event_date,10077,10077,0,0
8,integrity_cases,incident_date,1384,1384,0,0
9,integrity_cases,case_open_date,1384,1384,0,0


In [28]:
# ============================================================
# REMAINING DATETIME ISSUES
# ============================================================

remaining_invalid_dates = (
    datetime_cleaning_summary[
        datetime_cleaning_summary[
            "invalid_datetime_values"
        ] > 0
    ]
)

print(
    "Columns with unparseable datetime values:",
    len(remaining_invalid_dates)
)

remaining_invalid_dates

Columns with unparseable datetime values: 0


,table_name,column_name,values_before,valid_datetime_values,invalid_datetime_values,missing_values


In [29]:
# ============================================================
# INSPECT UNPARSEABLE DATETIME VALUES
# ============================================================

for table_name, columns in DATE_COLUMNS.items():

    df = clean_data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        # Re-read the corresponding raw values
        raw_df = data[table_name]

        raw_values = raw_df[column]

        parsed_values = pd.to_datetime(
            raw_values,
            format="mixed",
            errors="coerce"
        )

        invalid_mask = (
            parsed_values.isna()
            &
            raw_values.notna()
            &
            (
                raw_values
                .astype("string")
                .str.strip()
                != ""
            )
        )

        if invalid_mask.any():

            print(
                "\n" + "=" * 70
            )

            print(
                f"{table_name}.{column}"
            )

            print(
                "=" * 70
            )

            display(
                raw_values[
                    invalid_mask
                ].head(20)
            )

In [30]:
# ============================================================
# VERIFY DATETIME DTYPES
# ============================================================

datetime_dtype_check = []

for table_name, columns in DATE_COLUMNS.items():

    df = clean_data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        datetime_dtype_check.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "dtype":
                str(df[column].dtype),

            "missing_values":
                int(df[column].isna().sum())
        })


datetime_dtype_check = pd.DataFrame(
    datetime_dtype_check
)

datetime_dtype_check

,table_name,column_name,dtype,missing_values
0,academic_terms,start_date,datetime64[us],0
1,academic_terms,end_date,datetime64[us],0
2,students,enrollment_date,datetime64[us],0
3,assignments,assignment_open_at,datetime64[us],0
4,assignments,submission_deadline,datetime64[us],0
5,submissions,submitted_at,datetime64[ns],0
6,ai_detector_results,test_timestamp,datetime64[ns],0
7,student_behavior_events,event_date,datetime64[ns],0
8,integrity_cases,incident_date,datetime64[ns],0
9,integrity_cases,case_open_date,datetime64[ns],0


In [31]:
# ============================================================
# TERM TIMELINE VALIDATION
# ============================================================

term_timeline_errors = {

    "end_before_start": 0,
    "overlapping_terms": 0
}


# End must be after start
term_timeline_errors[
    "end_before_start"
] = int(
    (
        clean_data["academic_terms"]["end_date"]
        <=
        clean_data["academic_terms"]["start_date"]
    ).sum()
)


# Check overlapping terms
terms_sorted = (
    clean_data["academic_terms"]
    .sort_values("start_date")
    .reset_index(drop=True)
)


overlap_count = 0

for i in range(
    len(terms_sorted) - 1
):

    current_end = terms_sorted.loc[
        i,
        "end_date"
    ]

    next_start = terms_sorted.loc[
        i + 1,
        "start_date"
    ]

    if next_start <= current_end:
        overlap_count += 1


term_timeline_errors[
    "overlapping_terms"
] = overlap_count


print(
    "Term end-before-start errors:",
    term_timeline_errors[
        "end_before_start"
    ]
)

print(
    "Overlapping term pairs:",
    term_timeline_errors[
        "overlapping_terms"
    ]
)

Term end-before-start errors: 0
Overlapping term pairs: 1


In [32]:
# ============================================================
# ASSIGNMENT TIMELINE VALIDATION
# ============================================================

assignment_time = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left"
    )
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left"
    )
)


assignment_timeline_errors = {
    "missing_term": 0,
    "open_before_term": 0,
    "open_after_term": 0,
    "deadline_before_open": 0,
    "deadline_after_term": 0
}


assignment_timeline_errors[
    "missing_term"
] = int(
    assignment_time["start_date"].isna().sum()
)

assignment_timeline_errors[
    "open_before_term"
] = int(
    (
        assignment_time["assignment_open_at"]
        <
        assignment_time["start_date"]
    ).sum()
)

assignment_timeline_errors[
    "open_after_term"
] = int(
    (
        assignment_time["assignment_open_at"]
        >
        assignment_time["end_date"]
    ).sum()
)

assignment_timeline_errors[
    "deadline_before_open"
] = int(
    (
        assignment_time["submission_deadline"]
        <=
        assignment_time["assignment_open_at"]
    ).sum()
)

assignment_timeline_errors[
    "deadline_after_term"
] = int(
    (
        assignment_time["submission_deadline"]
        >
        assignment_time["end_date"]
    ).sum()
)


for issue, count in assignment_timeline_errors.items():

    print(
        f"{issue}: {count}"
    )

missing_term: 0
open_before_term: 0
open_after_term: 0
deadline_before_open: 5
deadline_after_term: 0


In [33]:
# ============================================================
# SUBMISSION TIMELINE VALIDATION
# ============================================================

submission_time = (
    clean_data["submissions"][
        [
            "submission_id",
            "student_id",
            "assignment_id",
            "submitted_at",
            "late_submission",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left"
    )
)


submission_errors = {

    "missing_assignment":
        0,

    "before_assignment_open":
        0,

    "late_flag_mismatch":
        0,

    "on_time_flag_mismatch":
        0,

    "hours_before_deadline_mismatch":
        0
}


submission_errors[
    "missing_assignment"
] = int(
    submission_time[
        "assignment_open_at"
    ].isna().sum()
)


submission_errors[
    "before_assignment_open"
] = int(
    (
        submission_time["submitted_at"]
        <
        submission_time["assignment_open_at"]
    ).sum()
)


late_flag_mismatch = (
    (
        submission_time[
            "late_submission"
        ].astype("boolean")
        == True
    )
    &
    (
        submission_time[
            "submitted_at"
        ]
        <=
        submission_time[
            "submission_deadline"
        ]
    )
)

on_time_flag_mismatch = (
    (
        submission_time[
            "late_submission"
        ].astype("boolean")
        == False
    )
    &
    (
        submission_time[
            "submitted_at"
        ]
        >
        submission_time[
            "submission_deadline"
        ]
    )
)


submission_errors[
    "late_flag_mismatch"
] = int(
    late_flag_mismatch.sum()
)

submission_errors[
    "on_time_flag_mismatch"
] = int(
    on_time_flag_mismatch.sum()
)


# Actual hour difference
calculated_hours = (
    (
        submission_time[
            "submission_deadline"
        ]
        -
        submission_time[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


hour_difference = (
    calculated_hours
    -
    pd.to_numeric(
        submission_time[
            "hours_before_deadline"
        ],
        errors="coerce"
    )
).abs()


submission_errors[
    "hours_before_deadline_mismatch"
] = int(
    (
        hour_difference
        > 0.05
    ).sum()
)


for issue, count in submission_errors.items():

    print(
        f"{issue}: {count}"
    )

missing_assignment: 0
before_assignment_open: 31
late_flag_mismatch: 8
on_time_flag_mismatch: 61
hours_before_deadline_mismatch: 125


In [34]:
# ============================================================
# DETECTOR TIMELINE VALIDATION
# ============================================================

detector_time = (
    clean_data["ai_detector_results"][
        [
            "detector_result_id",
            "submission_id",
            "tool_id",
            "test_timestamp"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "submitted_at"
            ]
        ],
        on="submission_id",
        how="left"
    )
)


detector_errors = {

    "missing_submission":
        0,

    "test_before_submission":
        0
}


detector_errors[
    "missing_submission"
] = int(
    detector_time[
        "submitted_at"
    ].isna().sum()
)

detector_errors[
    "test_before_submission"
] = int(
    (
        detector_time[
            "test_timestamp"
        ]
        <=
        detector_time[
            "submitted_at"
        ]
    ).sum()
)


for issue, count in detector_errors.items():

    print(
        f"{issue}: {count}"
    )

missing_submission: 0
test_before_submission: 249


In [35]:
# ============================================================
# BEHAVIOR EVENT TIMELINE VALIDATION
# ============================================================

behavior_time = (
    clean_data["student_behavior_events"][
        [
            "event_id",
            "student_id",
            "related_submission_id",
            "event_date"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        left_on=[
            "related_submission_id",
            "student_id"
        ],
        right_on=[
            "submission_id",
            "student_id"
        ],
        how="left"
    )
)


behavior_errors = {

    "missing_submission":
        0,

    "student_submission_mismatch":
        0,

    "event_before_submission":
        0
}


behavior_errors[
    "missing_submission"
] = int(
    behavior_time[
        "submission_id"
    ].isna().sum()
)


# Since we joined on student_id + submission_id,
# any successful match already confirms both.
behavior_errors[
    "student_submission_mismatch"
] = 0


behavior_errors[
    "event_before_submission"
] = int(
    (
        behavior_time["event_date"]
        <
        behavior_time["submitted_at"]
    ).sum()
)


for issue, count in behavior_errors.items():

    print(
        f"{issue}: {count}"
    )

missing_submission: 0
student_submission_mismatch: 0
event_before_submission: 39


In [36]:
# ============================================================
# INTEGRITY CASE TIMELINE VALIDATION
# ============================================================

case_time = (
    clean_data["integrity_cases"][
        [
            "case_id",
            "student_id",
            "submission_id",
            "incident_date",
            "case_open_date",
            "case_closed_date",
            "appeal_date",
            "verdict"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        on=[
            "submission_id",
            "student_id"
        ],
        how="left"
    )
)


case_errors = {

    "missing_submission":
        0,

    "incident_before_submission":
        0,

    "case_open_before_incident":
        0,

    "case_closed_before_open":
        0,

    "appeal_before_closure":
        0,

    "pending_with_closure":
        0
}


case_errors[
    "missing_submission"
] = int(
    case_time[
        "submitted_at"
    ].isna().sum()
)


case_errors[
    "incident_before_submission"
] = int(
    (
        case_time["incident_date"]
        <
        case_time["submitted_at"]
    ).sum()
)


case_errors[
    "case_open_before_incident"
] = int(
    (
        case_time["case_open_date"]
        <
        case_time["incident_date"]
    ).sum()
)


case_errors[
    "case_closed_before_open"
] = int(
    (
        case_time[
            "case_closed_date"
        ].notna()
        &
        (
            case_time[
                "case_closed_date"
            ]
            <
            case_time[
                "case_open_date"
            ]
        )
    ).sum()
)


case_errors[
    "appeal_before_closure"
] = int(
    (
        case_time["appeal_date"].notna()
        &
        case_time["case_closed_date"].notna()
        &
        (
            case_time[
                "appeal_date"
            ]
            <
            case_time[
                "case_closed_date"
            ]
        )
    ).sum()
)


case_errors[
    "pending_with_closure"
] = int(
    (
        (
            case_time["verdict"]
            == "Pending"
        )
        &
        case_time[
            "case_closed_date"
        ].notna()
    ).sum()
)


for issue, count in case_errors.items():

    print(
        f"{issue}: {count}"
    )

missing_submission: 0
incident_before_submission: 5
case_open_before_incident: 4
case_closed_before_open: 1
appeal_before_closure: 1
pending_with_closure: 0


In [37]:
# ============================================================
# INTEGRITY HISTORY TEMPORAL VALIDATION
# ============================================================

history = clean_data[
    "student_integrity_history"
]

cases = clean_data[
    "integrity_cases"
]

events = clean_data[
    "student_behavior_events"
]


# ------------------------------------------------------------
# Find any case occurring after a student's snapshot
# ------------------------------------------------------------

history_case_check = (
    history[
        [
            "student_id",
            "snapshot_at"
        ]
    ]
    .merge(
        cases[
            [
                "student_id",
                "case_open_date"
            ]
        ],
        on="student_id",
        how="left"
    )
)


future_case_mask = (
    history_case_check[
        "case_open_date"
    ].notna()
    &
    (
        history_case_check[
            "case_open_date"
        ]
        >=
        history_case_check[
            "snapshot_at"
        ]
    )
)


# ------------------------------------------------------------
# Find any event occurring after snapshot
# ------------------------------------------------------------

history_event_check = (
    history[
        [
            "student_id",
            "snapshot_at"
        ]
    ]
    .merge(
        events[
            [
                "student_id",
                "event_date"
            ]
        ],
        on="student_id",
        how="left"
    )
)


future_event_mask = (
    history_event_check[
        "event_date"
    ].notna()
    &
    (
        history_event_check[
            "event_date"
        ]
        >=
        history_event_check[
            "snapshot_at"
        ]
    )
)


print(
    "Future cases after snapshot:",
    int(future_case_mask.sum())
)

print(
    "Future events after snapshot:",
    int(future_event_mask.sum())
)

Future cases after snapshot: 4079
Future events after snapshot: 29272


In [38]:
# ============================================================
# SAVE DATETIME AUDIT REPORTS
# ============================================================

datetime_cleaning_summary.to_csv(
    AUDIT_DIR / "07_datetime_cleaning_summary.csv",
    index=False
)

datetime_dtype_check.to_csv(
    AUDIT_DIR / "08_datetime_dtype_check.csv",
    index=False
)

print(
    "Datetime audit reports saved successfully."
)

Datetime audit reports saved successfully.


In [39]:
# ============================================================
# DATETIME STANDARDIZATION
# ============================================================
#
# The raw datasets contain multiple datetime representations.
# We convert recognized values into pandas datetime objects.
#
# Unparseable values become NaT.
# We will decide which NaT values require row removal in the
# next step based on whether the field is mandatory or optional.
# ============================================================


datetime_parse_summary = []


for table_name, columns in DATE_COLUMNS.items():

    df = clean_data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        before = df[column].copy()

        parsed = pd.to_datetime(
            before,
            format="mixed",
            errors="coerce"
        )

        original_non_null = before.notna()

        invalid_count = int(
            (
                parsed.isna()
                &
                original_non_null
            ).sum()
        )

        missing_before = int(
            before.isna().sum()
        )

        clean_data[table_name][column] = parsed

        datetime_parse_summary.append({

            "table_name": table_name,
            "column_name": column,
            "original_missing": missing_before,
            "unparseable_values": invalid_count,
            "valid_datetime_values": int(
                parsed.notna().sum()
            ),
            "final_missing": int(
                parsed.isna().sum()
            )
        })


datetime_parse_summary = pd.DataFrame(
    datetime_parse_summary
)

datetime_parse_summary

,table_name,column_name,original_missing,unparseable_values,valid_datetime_values,final_missing
0,academic_terms,start_date,0,0,6,0
1,academic_terms,end_date,0,0,6,0
2,students,enrollment_date,0,0,1508,0
3,assignments,assignment_open_at,0,0,804,0
4,assignments,submission_deadline,0,0,804,0
5,submissions,submitted_at,0,0,15083,0
6,ai_detector_results,test_timestamp,0,0,75192,0
7,student_behavior_events,event_date,0,0,10077,0
8,integrity_cases,incident_date,0,0,1384,0
9,integrity_cases,case_open_date,0,0,1384,0


In [40]:
# ============================================================
# REQUIRED vs OPTIONAL DATETIME FIELDS
# ============================================================

REQUIRED_DATETIME_COLUMNS = {

    "academic_terms": [
        "start_date",
        "end_date"
    ],

    "students": [
        "enrollment_date"
    ],

    "assignments": [
        "assignment_open_at",
        "submission_deadline"
    ],

    "submissions": [
        "submitted_at"
    ],

    "ai_detector_results": [
        "test_timestamp"
    ],

    "student_behavior_events": [
        "event_date"
    ],

    "integrity_cases": [
        "incident_date",
        "case_open_date"
    ],

    "student_integrity_history": [
        "snapshot_at"
    ]
}


OPTIONAL_DATETIME_COLUMNS = {

    "integrity_cases": [
        "appeal_date",
        "case_closed_date"
    ]
}


print("Required datetime fields:")
for table, columns in REQUIRED_DATETIME_COLUMNS.items():
    print(f"{table}: {columns}")

print("\nOptional datetime fields:")
for table, columns in OPTIONAL_DATETIME_COLUMNS.items():
    print(f"{table}: {columns}")

Required datetime fields:
academic_terms: ['start_date', 'end_date']
students: ['enrollment_date']
assignments: ['assignment_open_at', 'submission_deadline']
submissions: ['submitted_at']
ai_detector_results: ['test_timestamp']
student_behavior_events: ['event_date']
integrity_cases: ['incident_date', 'case_open_date']
student_integrity_history: ['snapshot_at']

Optional datetime fields:
integrity_cases: ['appeal_date', 'case_closed_date']


In [41]:
# ============================================================
# DROP RECORDS WITH INVALID REQUIRED DATETIMES
# ============================================================

datetime_drop_log = []


for table_name, columns in REQUIRED_DATETIME_COLUMNS.items():

    df = clean_data[table_name].copy()

    rows_before = len(df)

    invalid_row_mask = pd.Series(
        False,
        index=df.index
    )

    for column in columns:

        if column not in df.columns:
            continue

        invalid_row_mask |= (
            df[column].isna()
        )


    rows_to_drop = int(
        invalid_row_mask.sum()
    )

    if rows_to_drop > 0:

        clean_data[table_name] = (
            df.loc[
                ~invalid_row_mask
            ]
            .reset_index(
                drop=True
            )
        )

        log_cleaning(
            table_name,
            "*",
            "Missing/unparseable required datetime",
            f"Dropped {rows_to_drop} records",
            "Required event timestamps could not be reliably reconstructed."
        )

    else:

        clean_data[table_name] = df


    datetime_drop_log.append({

        "table_name":
            table_name,

        "rows_before":
            rows_before,

        "rows_dropped":
            rows_to_drop,

        "rows_after":
            len(
                clean_data[table_name]
            )
    })


datetime_drop_log = pd.DataFrame(
    datetime_drop_log
)

datetime_drop_log

,table_name,rows_before,rows_dropped,rows_after
0,academic_terms,6,0,6
1,students,1508,0,1508
2,assignments,804,0,804
3,submissions,15083,0,15083
4,ai_detector_results,75192,0,75192
5,student_behavior_events,10077,0,10077
6,integrity_cases,1384,0,1384
7,student_integrity_history,7511,0,7511


In [42]:
# ============================================================
# DATETIME DTYPE CHECK
# ============================================================

datetime_dtype_check = []


for table_name, columns in DATE_COLUMNS.items():

    df = clean_data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        datetime_dtype_check.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "dtype":
                str(
                    df[column].dtype
                ),

            "missing_values":
                int(
                    df[column].isna().sum()
                )
        })


datetime_dtype_check = pd.DataFrame(
    datetime_dtype_check
)

datetime_dtype_check

,table_name,column_name,dtype,missing_values
0,academic_terms,start_date,datetime64[us],0
1,academic_terms,end_date,datetime64[us],0
2,students,enrollment_date,datetime64[us],0
3,assignments,assignment_open_at,datetime64[us],0
4,assignments,submission_deadline,datetime64[us],0
5,submissions,submitted_at,datetime64[ns],0
6,ai_detector_results,test_timestamp,datetime64[ns],0
7,student_behavior_events,event_date,datetime64[ns],0
8,integrity_cases,incident_date,datetime64[ns],0
9,integrity_cases,case_open_date,datetime64[ns],0


In [43]:
# ============================================================
# ASSIGNMENT TIMELINE VALIDATION
# ============================================================

assignment_time = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left"
    )
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left"
    )
)


assignment_errors = {

    "missing_term": int(
        assignment_time["start_date"].isna().sum()
    ),

    "opening_before_term": int(
        (
            assignment_time["assignment_open_at"]
            <
            assignment_time["start_date"]
        ).sum()
    ),

    "opening_after_term": int(
        (
            assignment_time["assignment_open_at"]
            >
            assignment_time["end_date"]
        ).sum()
    ),

    "deadline_before_open": int(
        (
            assignment_time["submission_deadline"]
            <=
            assignment_time["assignment_open_at"]
        ).sum()
    ),

    "deadline_after_term": int(
        (
            assignment_time["submission_deadline"]
            >
            assignment_time["end_date"]
        ).sum()
    )
}


print("Assignment timeline validation:")

for issue, count in assignment_errors.items():
    print(
        f"{issue}: {count}"
    )

Assignment timeline validation:
missing_term: 0
opening_before_term: 0
opening_after_term: 0
deadline_before_open: 5
deadline_after_term: 0


In [44]:
# ============================================================
# SUBMISSION TIMELINE VALIDATION
# ============================================================

submission_time = (
    clean_data["submissions"][
        [
            "submission_id",
            "student_id",
            "assignment_id",
            "submitted_at",
            "late_submission",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left"
    )
)


submission_errors = {

    "missing_assignment":
        int(
            submission_time[
                "assignment_open_at"
            ].isna().sum()
        ),

    "before_assignment_open":
        int(
            (
                submission_time["submitted_at"]
                <
                submission_time["assignment_open_at"]
            ).sum()
        ),

    "late_flag_mismatch":
        int(
            (
                (
                    submission_time[
                        "late_submission"
                    ]
                    == True
                )
                &
                (
                    submission_time[
                        "submitted_at"
                    ]
                    <=
                    submission_time[
                        "submission_deadline"
                    ]
                )
            ).sum()
        ),

    "on_time_flag_mismatch":
        int(
            (
                (
                    submission_time[
                        "late_submission"
                    ]
                    == False
                )
                &
                (
                    submission_time[
                        "submitted_at"
                    ]
                    >
                    submission_time[
                        "submission_deadline"
                    ]
                )
            ).sum()
        )
    }


# Hours before deadline
calculated_hours = (
    (
        submission_time[
            "submission_deadline"
        ]
        -
        submission_time[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)

stored_hours = pd.to_numeric(
    submission_time[
        "hours_before_deadline"
    ],
    errors="coerce"
)

hours_difference = (
    calculated_hours
    -
    stored_hours
).abs()


submission_errors[
    "hours_before_deadline_mismatch"
] = int(
    (
        hours_difference > 0.05
    ).sum()
)


print("Submission timeline validation:")

for issue, count in submission_errors.items():
    print(
        f"{issue}: {count}"
    )

Submission timeline validation:
missing_assignment: 0
before_assignment_open: 31
late_flag_mismatch: 8
on_time_flag_mismatch: 61
hours_before_deadline_mismatch: 125


In [45]:
# ============================================================
# PRIMARY-KEY DEDUPLICATION
# ============================================================
#
# Exact duplicates were already removed earlier.
#
# Some duplicate primary-key records remain because the raw
# corruption process altered formatting/values in duplicated
# rows, making them no longer exact duplicates.
#
# Cleaning rule:
# - Keep the first record for each primary key.
# - Drop later duplicate-key records.
#
# This prevents one-to-many explosions during relational joins
# and establishes one authoritative record per entity.
# ============================================================


primary_key_drop_log = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    df = clean_data[table_name].copy()

    rows_before = len(df)

    duplicate_mask = (
        df.duplicated(
            subset=pk_columns,
            keep="first"
        )
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

    if duplicate_count > 0:

        df = (
            df.loc[
                ~duplicate_mask
            ]
            .reset_index(
                drop=True
            )
        )

        clean_data[table_name] = df

        log_cleaning(
            table_name,
            ", ".join(pk_columns),
            "Duplicate primary-key records",
            f"Dropped {duplicate_count} duplicate-key records",
            "One authoritative record is retained per primary key."
        )

    else:

        clean_data[table_name] = df


    primary_key_drop_log.append({

        "table_name":
            table_name,

        "primary_key":
            ", ".join(pk_columns),

        "rows_before":
            rows_before,

        "duplicate_key_rows_dropped":
            duplicate_count,

        "rows_after":
            len(df)
    })


primary_key_drop_log = pd.DataFrame(
    primary_key_drop_log
)

primary_key_drop_log

,table_name,primary_key,rows_before,duplicate_key_rows_dropped,rows_after
0,academic_terms,term_id,6,1,5
1,departments,department_id,10,0,10
2,majors,major_id,24,0,24
3,instructors,instructor_id,100,0,100
4,students,student_id,1508,8,1500
5,courses,course_id,90,0,90
6,course_offerings,offering_id,261,0,261
7,assignments,assignment_id,804,4,800
8,submissions,submission_id,15083,83,15000
9,detector_tools,tool_id,5,0,5


In [46]:
# ============================================================
# PRIMARY-KEY VALIDATION AFTER DEDUPLICATION
# ============================================================

pk_validation_rows = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    df = clean_data[table_name]

    null_count = int(
        df[pk_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    duplicate_count = int(
        df.duplicated(
            subset=pk_columns,
            keep=False
        ).sum()
    )

    pk_validation_rows.append({

        "table_name":
            table_name,

        "primary_key":
            ", ".join(pk_columns),

        "rows":
            len(df),

        "null_key_rows":
            null_count,

        "duplicate_key_rows":
            duplicate_count
    })


pk_validation = pd.DataFrame(
    pk_validation_rows
)

pk_validation

,table_name,primary_key,rows,null_key_rows,duplicate_key_rows
0,academic_terms,term_id,5,0,0
1,departments,department_id,10,0,0
2,majors,major_id,24,0,0
3,instructors,instructor_id,100,0,0
4,students,student_id,1500,0,0
5,courses,course_id,90,0,0
6,course_offerings,offering_id,261,0,0
7,assignments,assignment_id,800,0,0
8,submissions,submission_id,15000,0,0
9,detector_tools,tool_id,5,0,0


In [47]:
# ============================================================
# ASSIGNMENT TIMELINE RECHECK
# ============================================================

assignment_time = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Assignment rows after joins:",
    len(assignment_time)
)


assignment_errors = {

    "missing_term": int(
        assignment_time[
            "start_date"
        ].isna().sum()
    ),

    "opening_before_term": int(
        (
            assignment_time[
                "assignment_open_at"
            ]
            <
            assignment_time[
                "start_date"
            ]
        ).sum()
    ),

    "opening_after_term": int(
        (
            assignment_time[
                "assignment_open_at"
            ]
            >
            assignment_time[
                "end_date"
            ]
        ).sum()
    ),

    "deadline_before_open": int(
        (
            assignment_time[
                "submission_deadline"
            ]
            <=
            assignment_time[
                "assignment_open_at"
            ]
        ).sum()
    ),

    "deadline_after_term": int(
        (
            assignment_time[
                "submission_deadline"
            ]
            >
            assignment_time[
                "end_date"
            ]
        ).sum()
    )
}


for issue, count in assignment_errors.items():

    print(
        f"{issue}: {count}"
    )

Assignment rows after joins: 800
missing_term: 0
opening_before_term: 0
opening_after_term: 0
deadline_before_open: 4
deadline_after_term: 0


In [48]:
# ============================================================
# SUBMISSION TIMELINE RECHECK
# ============================================================

submission_time = (
    clean_data["submissions"][
        [
            "submission_id",
            "student_id",
            "assignment_id",
            "submitted_at",
            "late_submission",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Submission rows after join:",
    len(submission_time)
)


print(
    "Missing assignment:",
    submission_time[
        "assignment_open_at"
    ].isna().sum()
)


print(
    "Before assignment open:",
    (
        submission_time[
            "submitted_at"
        ]
        <
        submission_time[
            "assignment_open_at"
        ]
    ).sum()
)


late_flag_mismatch = (
    (
        submission_time[
            "late_submission"
        ]
        == True
    )
    &
    (
        submission_time[
            "submitted_at"
        ]
        <=
        submission_time[
            "submission_deadline"
        ]
    )
)


on_time_flag_mismatch = (
    (
        submission_time[
            "late_submission"
        ]
        == False
    )
    &
    (
        submission_time[
            "submitted_at"
        ]
        >
        submission_time[
            "submission_deadline"
        ]
    )
)


print(
    "Late flag mismatch:",
    late_flag_mismatch.sum()
)

print(
    "On-time flag mismatch:",
    on_time_flag_mismatch.sum()
)


calculated_hours = (
    (
        submission_time[
            "submission_deadline"
        ]
        -
        submission_time[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


stored_hours = pd.to_numeric(
    submission_time[
        "hours_before_deadline"
    ],
    errors="coerce"
)


hour_difference = (
    calculated_hours
    -
    stored_hours
).abs()


print(
    "Hours-before-deadline mismatch:",
    (
        hour_difference > 0.05
    ).sum()
)

Submission rows after join: 15000
Missing assignment: 0
Before assignment open: 30
Late flag mismatch: 8
On-time flag mismatch: 61
Hours-before-deadline mismatch: 124


In [49]:
# ============================================================
# DATETIME CANDIDATE PARSER
# ============================================================
#
# The raw datasets contain several legitimate datetime
# representations. This helper returns every interpretation
# that can be parsed successfully.
#
# Later, relational/business rules determine which candidate
# is actually valid.
# ============================================================


def parse_datetime_candidates(value):

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "":
        return []

    formats = [

        # ISO-style formats
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",

        # Slash-separated ISO-style formats
        "%Y/%m/%d %H:%M:%S.%f",
        "%Y/%m/%d %H:%M:%S",
        "%Y/%m/%d %H:%M",

        # Day-month-year formats
        "%d-%m-%Y %H:%M:%S",
        "%d-%m-%Y %H:%M",

        # Month-day-year formats
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",

        # Month/day/year formats
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",

        # Day/month/year formats
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",

        # Date-only formats
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d-%m-%Y",
        "%m-%d-%Y",
        "%d/%m/%Y",
        "%m/%d/%Y"
    ]

    candidates = []

    for fmt in formats:

        try:

            parsed = pd.to_datetime(
                text,
                format=fmt
            )

            parsed = pd.Timestamp(
                parsed
            )

            # Avoid duplicate interpretations
            if parsed not in candidates:
                candidates.append(
                    parsed
                )

        except (
            ValueError,
            TypeError,
            OverflowError
        ):
            continue

    return candidates


print(
    "Datetime candidate parser loaded successfully."
)

# Quick test
test_value = "12/09/2026 11:00"

print(
    f"\nTest value: {test_value}"
)

print(
    "Possible interpretations:"
)

for candidate in parse_datetime_candidates(
    test_value
):
    print(
        " ",
        candidate
    )

Datetime candidate parser loaded successfully.

Test value: 12/09/2026 11:00
Possible interpretations:
  2026-12-09 11:00:00
  2026-09-12 11:00:00


In [50]:
# ============================================================
# CONTEXT-AWARE ASSIGNMENT DATETIME REPAIR
# ============================================================
#
# Assignment dates are resolved using the academic term
# attached to each course offering.
#
# A date interpretation is accepted only when:
#
# term_start
#     <=
# assignment_open_at
#     <
# submission_deadline
#     <=
# term_end
#
# This prevents ambiguous DD/MM vs MM/DD parsing from creating
# impossible assignment timelines.
# ============================================================


def resolve_assignment_datetime(
    raw_value,
    term_start,
    term_end,
    other_value=None,
    field="open"
):

    candidates = parse_datetime_candidates(
        raw_value
    )

    if not candidates:
        return pd.NaT

    valid_candidates = []

    for candidate in candidates:

        candidate = pd.Timestamp(
            candidate
        )

        if field == "open":

            # If deadline is available, the opening must
            # occur before it.
            if other_value is not None:

                if (
                    candidate >= term_start
                    and candidate < other_value
                    and candidate <= term_end
                ):
                    valid_candidates.append(
                        candidate
                    )

            else:

                if (
                    candidate >= term_start
                    and candidate <= term_end
                ):
                    valid_candidates.append(
                        candidate
                    )

        elif field == "deadline":

            # If opening is available, the deadline must
            # occur after it.
            if other_value is not None:

                if (
                    candidate > other_value
                    and candidate <= term_end
                    and candidate >= term_start
                ):
                    valid_candidates.append(
                        candidate
                    )

            else:

                if (
                    candidate >= term_start
                    and candidate <= term_end
                ):
                    valid_candidates.append(
                        candidate
                    )

    if len(valid_candidates) == 1:

        return valid_candidates[0]

    # If multiple candidates remain, keep the first
    # only when all are within the same valid timeline.
    if len(valid_candidates) > 1:

        return sorted(
            valid_candidates
        )[0]

    return pd.NaT


# ============================================================
# BUILD ASSIGNMENT CONTEXT
# ============================================================

assignment_context = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left",
        validate="many_to_one"
    )
)


# ============================================================
# RAW ASSIGNMENT LOOKUP
# ============================================================

raw_assignment_lookup = (
    data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .copy()
)

raw_assignment_lookup["assignment_id"] = (
    raw_assignment_lookup["assignment_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

raw_assignment_lookup["offering_id"] = (
    raw_assignment_lookup["offering_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ============================================================
# RESOLVE PROBLEMATIC ASSIGNMENTS
# ============================================================

assignment_repairs = []


for idx, row in assignment_context.iterrows():

    assignment_id = row["assignment_id"]
    offering_id = row["offering_id"]

    term_start = pd.Timestamp(
        row["start_date"]
    )

    term_end = pd.Timestamp(
        row["end_date"]
    )

    current_open = row[
        "assignment_open_at"
    ]

    current_deadline = row[
        "submission_deadline"
    ]


    # Only inspect records currently violating the
    # assignment timeline.
    currently_invalid = (
        pd.isna(current_open)
        or
        pd.isna(current_deadline)
        or
        current_open < term_start
        or
        current_open > term_end
        or
        current_deadline <= current_open
        or
        current_deadline > term_end
    )

    if not currently_invalid:
        continue


    raw_match = raw_assignment_lookup[
        (
            raw_assignment_lookup[
                "assignment_id"
            ]
            == assignment_id
        )
        &
        (
            raw_assignment_lookup[
                "offering_id"
            ]
            == offering_id
        )
    ]


    if raw_match.empty:
        continue


    raw_row = raw_match.iloc[0]

    open_candidates = (
        parse_datetime_candidates(
            raw_row[
                "assignment_open_at"
            ]
        )
    )

    deadline_candidates = (
        parse_datetime_candidates(
            raw_row[
                "submission_deadline"
            ]
        )
    )


    valid_pairs = []

    for opening in open_candidates:

        for deadline in deadline_candidates:

            opening = pd.Timestamp(
                opening
            )

            deadline = pd.Timestamp(
                deadline
            )

            if (
                opening >= term_start
                and
                opening < deadline
                and
                deadline <= term_end
            ):

                valid_pairs.append(
                    (
                        opening,
                        deadline
                    )
                )


    if len(valid_pairs) >= 1:

        # Prefer a unique valid pair.
        resolved_opening, resolved_deadline = (
            sorted(valid_pairs)[0]
        )

        old_open = clean_data[
            "assignments"
        ].loc[
            clean_data[
                "assignments"
            ]["assignment_id"]
            == assignment_id,
            "assignment_open_at"
        ].iloc[0]

        old_deadline = clean_data[
            "assignments"
        ].loc[
            clean_data[
                "assignments"
            ]["assignment_id"]
            == assignment_id,
            "submission_deadline"
        ].iloc[0]


        clean_data[
            "assignments"
        ].loc[
            clean_data[
                "assignments"
            ]["assignment_id"]
            == assignment_id,
            "assignment_open_at"
        ] = resolved_opening

        clean_data[
            "assignments"
        ].loc[
            clean_data[
                "assignments"
            ]["assignment_id"]
            == assignment_id,
            "submission_deadline"
        ] = resolved_deadline


        assignment_repairs.append({

            "assignment_id":
                assignment_id,

            "old_open":
                old_open,

            "new_open":
                resolved_opening,

            "old_deadline":
                old_deadline,

            "new_deadline":
                resolved_deadline,

            "repair_status":
                "Resolved using term context"
        })


assignment_repairs = pd.DataFrame(
    assignment_repairs
)


print(
    "Assignment datetime repairs:",
    len(assignment_repairs)
)

if not assignment_repairs.empty:
    display(
        assignment_repairs
    )

Assignment datetime repairs: 4


,assignment_id,old_open,new_open,old_deadline,new_deadline,repair_status
0,ASSIGN00041,2026-09-28 18:00:00,2026-09-28 18:00:00,2026-01-11 04:00:00,2026-11-01 04:00:00,Resolved using term context
1,ASSIGN00049,2026-12-09 11:00:00,2026-09-12 11:00:00,2026-10-15 04:00:00,2026-10-15 04:00:00,Resolved using term context
2,ASSIGN00458,2026-03-07 17:00:00,2026-03-07 17:00:00,2026-02-04 06:00:00,2026-04-02 06:00:00,Resolved using term context
3,ASSIGN00740,2026-10-16 13:00:00,2026-10-16 13:00:00,2026-06-11 04:00:00,2026-11-06 04:00:00,Resolved using term context


In [51]:
# ============================================================
# ASSIGNMENT TIMELINE RECHECK
# ============================================================

assignment_time = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id",
            "assignment_open_at",
            "submission_deadline"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left",
        validate="many_to_one"
    )
)


assignment_errors = {

    "missing_term": int(
        assignment_time[
            "start_date"
        ].isna().sum()
    ),

    "opening_before_term": int(
        (
            assignment_time[
                "assignment_open_at"
            ]
            <
            assignment_time[
                "start_date"
            ]
        ).sum()
    ),

    "opening_after_term": int(
        (
            assignment_time[
                "assignment_open_at"
            ]
            >
            assignment_time[
                "end_date"
            ]
        ).sum()
    ),

    "deadline_before_open": int(
        (
            assignment_time[
                "submission_deadline"
            ]
            <=
            assignment_time[
                "assignment_open_at"
            ]
        ).sum()
    ),

    "deadline_after_term": int(
        (
            assignment_time[
                "submission_deadline"
            ]
            >
            assignment_time[
                "end_date"
            ]
        ).sum()
    )
}


print("Assignment timeline validation:")

for issue, count in assignment_errors.items():

    print(
        f"{issue}: {count}"
    )

Assignment timeline validation:
missing_term: 0
opening_before_term: 0
opening_after_term: 0
deadline_before_open: 0
deadline_after_term: 0


In [52]:
# ============================================================
# RECALCULATE SUBMISSION TIMING FIELDS
# ============================================================
#
# late_submission and hours_before_deadline are derived
# variables. We calculate them from submission_deadline and
# submitted_at instead of trusting potentially corrupted
# source values.
# ============================================================


submission_time = (
    clean_data["submissions"][
        [
            "submission_id",
            "student_id",
            "assignment_id",
            "submitted_at",
            "late_submission",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# Identify submissions that occur before assignment opening
# ------------------------------------------------------------

invalid_submission_timing = (
    submission_time[
        "submitted_at"
    ]
    <
    submission_time[
        "assignment_open_at"
    ]
)


print(
    "Submissions before assignment opening:",
    int(
        invalid_submission_timing.sum()
    )
)


# ------------------------------------------------------------
# Recalculate derived fields
# ------------------------------------------------------------

calculated_hours = (
    (
        submission_time[
            "submission_deadline"
        ]
        -
        submission_time[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


calculated_late = (
    calculated_hours < 0
)


# ------------------------------------------------------------
# Update original dataframe
# ------------------------------------------------------------

submission_ids = submission_time[
    "submission_id"
].values


recalculated_values = pd.DataFrame({

    "submission_id":
        submission_ids,

    "new_hours_before_deadline":
        calculated_hours,

    "new_late_submission":
        calculated_late
})


clean_data["submissions"] = (
    clean_data["submissions"]
    .merge(
        recalculated_values,
        on="submission_id",
        how="left",
        validate="one_to_one"
    )
)


clean_data["submissions"][
    "hours_before_deadline"
] = (
    clean_data["submissions"][
        "new_hours_before_deadline"
    ]
    .round(2)
)


clean_data["submissions"][
    "late_submission"
] = (
    clean_data["submissions"][
        "new_late_submission"
    ]
    .astype("boolean")
)


clean_data["submissions"].drop(
    columns=[
        "new_hours_before_deadline",
        "new_late_submission"
    ],
    inplace=True
)


log_cleaning(
    "submissions",
    "late_submission",
    "Source value may be inconsistent with timestamps",
    "Recalculated from submitted_at and submission_deadline",
    "Late status is a deterministic derived field."
)

log_cleaning(
    "submissions",
    "hours_before_deadline",
    "Source value may be inconsistent with timestamps",
    "Recalculated from submitted_at and submission_deadline",
    "Time-to-deadline is a deterministic derived field."
)

Submissions before assignment opening: 17


In [53]:
# ============================================================
# DROP SUBMISSIONS WITH IMPOSSIBLE TIMING
# ============================================================

submission_time = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id",
            "submitted_at"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


invalid_submission_ids = (
    submission_time.loc[
        submission_time[
            "submitted_at"
        ]
        <
        submission_time[
            "assignment_open_at"
        ],
        "submission_id"
    ]
    .tolist()
)


print(
    "Invalid submissions identified:",
    len(invalid_submission_ids)
)


if invalid_submission_ids:

    clean_data["submissions"] = (
        clean_data["submissions"][
            ~clean_data["submissions"][
                "submission_id"
            ].isin(
                invalid_submission_ids
            )
        ]
        .reset_index(drop=True)
    )


    log_cleaning(
        "submissions",
        "submitted_at",
        "Submission occurred before assignment opened",
        f"Dropped {len(invalid_submission_ids)} submissions",
        "The timestamp could not be safely reconstructed without inventing information."
    )


print(
    "Submissions remaining:",
    len(clean_data["submissions"])
)

Invalid submissions identified: 17
Submissions remaining: 14983


In [54]:
# ============================================================
# CASCADE REMOVAL FOR INVALID SUBMISSIONS
# ============================================================

removed_submission_set = set(
    invalid_submission_ids
)


# ------------------------------------------------------------
# Detector results
# ------------------------------------------------------------

before = len(
    clean_data["ai_detector_results"]
)

clean_data["ai_detector_results"] = (
    clean_data["ai_detector_results"][
        ~clean_data[
            "ai_detector_results"
        ][
            "submission_id"
        ].isin(
            removed_submission_set
        )
    ]
    .reset_index(drop=True)
)

removed = (
    before
    - len(
        clean_data[
            "ai_detector_results"
        ]
    )
)

if removed > 0:

    log_cleaning(
        "ai_detector_results",
        "submission_id",
        "Parent submission removed",
        f"Dropped {removed} dependent detector records",
        "Prevents orphaned detector results."
    )


# ------------------------------------------------------------
# Behavior events
# ------------------------------------------------------------

before = len(
    clean_data[
        "student_behavior_events"
    ]
)

clean_data[
    "student_behavior_events"
] = (
    clean_data[
        "student_behavior_events"
    ][
        ~clean_data[
            "student_behavior_events"
        ][
            "related_submission_id"
        ].isin(
            removed_submission_set
        )
    ]
    .reset_index(drop=True)
)

removed = (
    before
    -
    len(
        clean_data[
            "student_behavior_events"
        ]
    )
)

if removed > 0:

    log_cleaning(
        "student_behavior_events",
        "related_submission_id",
        "Parent submission removed",
        f"Dropped {removed} dependent behavior events",
        "Prevents orphaned event records."
    )


# ------------------------------------------------------------
# Integrity cases
# ------------------------------------------------------------

before = len(
    clean_data[
        "integrity_cases"
    ]
)

clean_data[
    "integrity_cases"
] = (
    clean_data[
        "integrity_cases"
    ][
        ~clean_data[
            "integrity_cases"
        ][
            "submission_id"
        ].isin(
            removed_submission_set
        )
    ]
    .reset_index(drop=True)
)

removed = (
    before
    -
    len(
        clean_data[
            "integrity_cases"
        ]
    )
)

if removed > 0:

    log_cleaning(
        "integrity_cases",
        "submission_id",
        "Parent submission removed",
        f"Dropped {removed} dependent integrity cases",
        "Prevents orphaned case records."
    )


print(
    "\nCascade cleanup completed."
)


Cascade cleanup completed.


In [55]:
# ============================================================
# FINAL SUBMISSION TIMELINE CHECK
# ============================================================

submission_check = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id",
            "submitted_at",
            "late_submission",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


before_open = (
    submission_check[
        "submitted_at"
    ]
    <
    submission_check[
        "assignment_open_at"
    ]
).sum()


late_mismatch = (
    submission_check[
        "late_submission"
    ]
    !=
    (
        submission_check[
            "submitted_at"
        ]
        >
        submission_check[
            "submission_deadline"
        ]
    )
).sum()


calculated_hours = (
    (
        submission_check[
            "submission_deadline"
        ]
        -
        submission_check[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


hours_mismatch = (
    (
        calculated_hours
        -
        submission_check[
            "hours_before_deadline"
        ]
    ).abs()
    > 0.05
).sum()


print(
    "Submissions before assignment opening:",
    before_open
)

print(
    "Late-status mismatches:",
    late_mismatch
)

print(
    "Hours-before-deadline mismatches:",
    hours_mismatch
)

Submissions before assignment opening: 0
Late-status mismatches: 0
Hours-before-deadline mismatches: 0


In [56]:
# ============================================================
# AI DETECTOR TIMELINE VALIDATION
# ============================================================
#
# A detector can only test a submission after that submission
# has been submitted.
#
# Required relationship:
#
# submitted_at < test_timestamp
#
# Invalid detector-result records will be dropped because
# their timestamps cannot be reliably reconstructed.
# ============================================================


detector_time = (
    clean_data["ai_detector_results"][
        [
            "detector_result_id",
            "submission_id",
            "tool_id",
            "test_timestamp"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "submitted_at"
            ]
        ],
        on="submission_id",
        how="left",
        validate="many_to_one"
    )
)


detector_invalid_mask = (
    detector_time["submitted_at"].isna()
    |
    detector_time["test_timestamp"].isna()
    |
    (
        detector_time["test_timestamp"]
        <=
        detector_time["submitted_at"]
    )
)


print(
    "Detector records:",
    len(detector_time)
)

print(
    "Invalid detector timeline records:",
    int(detector_invalid_mask.sum())
)

Detector records: 74915
Invalid detector timeline records: 246


In [57]:
# ============================================================
# DROP INVALID DETECTOR TIMESTAMPS
# ============================================================

invalid_detector_ids = (
    detector_time.loc[
        detector_invalid_mask,
        "detector_result_id"
    ]
    .tolist()
)


if invalid_detector_ids:

    clean_data["ai_detector_results"] = (
        clean_data["ai_detector_results"][
            ~clean_data[
                "ai_detector_results"
            ][
                "detector_result_id"
            ].isin(
                invalid_detector_ids
            )
        ]
        .reset_index(drop=True)
    )

    log_cleaning(
        "ai_detector_results",
        "test_timestamp",
        "Detector test occurred before or at submission",
        f"Dropped {len(invalid_detector_ids)} detector results",
        "Detector results with impossible chronology cannot be reliably reconstructed."
    )


print(
    "Detector results remaining:",
    len(
        clean_data[
            "ai_detector_results"
        ]
    )
)

Detector results remaining: 74669


In [58]:
# ============================================================
# BEHAVIOR EVENT TIMELINE VALIDATION
# ============================================================

behavior_time = (
    clean_data["student_behavior_events"][
        [
            "event_id",
            "student_id",
            "related_submission_id",
            "event_date"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        left_on=[
            "related_submission_id",
            "student_id"
        ],
        right_on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)


behavior_invalid_mask = (
    behavior_time["submission_id"].isna()
    |
    behavior_time["event_date"].isna()
    |
    (
        behavior_time["event_date"]
        <
        behavior_time["submitted_at"]
    )
)


print(
    "Behavior events:",
    len(behavior_time)
)

print(
    "Invalid behavior-event records:",
    int(
        behavior_invalid_mask.sum()
    )
)

Behavior events: 10023
Invalid behavior-event records: 39


In [59]:
# ============================================================
# BEHAVIOR EVENT TIMELINE VALIDATION
# ============================================================

behavior_time = (
    clean_data["student_behavior_events"][
        [
            "event_id",
            "student_id",
            "related_submission_id",
            "event_date"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        left_on=[
            "related_submission_id",
            "student_id"
        ],
        right_on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)


behavior_invalid_mask = (
    behavior_time["submission_id"].isna()
    |
    behavior_time["event_date"].isna()
    |
    (
        behavior_time["event_date"]
        <
        behavior_time["submitted_at"]
    )
)


print(
    "Behavior events:",
    len(behavior_time)
)

print(
    "Invalid behavior-event records:",
    int(
        behavior_invalid_mask.sum()
    )
)

Behavior events: 10023
Invalid behavior-event records: 39


In [60]:
# ============================================================
# DROP INVALID BEHAVIOR EVENTS
# ============================================================

invalid_event_ids = (
    behavior_time.loc[
        behavior_invalid_mask,
        "event_id"
    ]
    .tolist()
)


if invalid_event_ids:

    clean_data[
        "student_behavior_events"
    ] = (
        clean_data[
            "student_behavior_events"
        ][
            ~clean_data[
                "student_behavior_events"
            ][
                "event_id"
            ].isin(
                invalid_event_ids
            )
        ]
        .reset_index(drop=True)
    )

    log_cleaning(
        "student_behavior_events",
        "event_date",
        "Behavior event occurred before related submission",
        f"Dropped {len(invalid_event_ids)} behavior events",
        "The event timestamp could not be reliably reconstructed."
    )


print(
    "Behavior events remaining:",
    len(
        clean_data[
            "student_behavior_events"
        ]
    )
)

Behavior events remaining: 9984


In [61]:
# ============================================================
# INTEGRITY CASE TIMELINE VALIDATION
# ============================================================

case_time = (
    clean_data["integrity_cases"][
        [
            "case_id",
            "student_id",
            "submission_id",
            "incident_date",
            "case_open_date",
            "case_closed_date",
            "appeal_date",
            "verdict"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)


case_errors = {

    "missing_submission":
        case_time[
            "submitted_at"
        ].isna(),

    "incident_before_submission":
        (
            case_time["incident_date"]
            <
            case_time["submitted_at"]
        ),

    "case_open_before_incident":
        (
            case_time["case_open_date"]
            <
            case_time["incident_date"]
        ),

    "case_closed_before_open":
        (
            case_time["case_closed_date"].notna()
            &
            (
                case_time["case_closed_date"]
                <
                case_time["case_open_date"]
            )
        ),

    "appeal_before_closure":
        (
            case_time["appeal_date"].notna()
            &
            case_time["case_closed_date"].notna()
            &
            (
                case_time["appeal_date"]
                <
                case_time["case_closed_date"]
            )
        )
}


for issue, mask in case_errors.items():

    print(
        f"{issue}: {int(mask.sum())}"
    )

missing_submission: 0
incident_before_submission: 5
case_open_before_incident: 4
case_closed_before_open: 1
appeal_before_closure: 1


In [62]:
# ============================================================
# DROP CASES WITH IMPOSSIBLE CHRONOLOGY
# ============================================================

invalid_case_mask = (
    case_time["submitted_at"].isna()
    |
    case_time["incident_date"].isna()
    |
    case_time["case_open_date"].isna()
    |
    (
        case_time["incident_date"]
        <
        case_time["submitted_at"]
    )
    |
    (
        case_time["case_open_date"]
        <
        case_time["incident_date"]
    )
    |
    (
        case_time["case_closed_date"].notna()
        &
        (
            case_time["case_closed_date"]
            <
            case_time["case_open_date"]
        )
    )
    |
    (
        case_time["appeal_date"].notna()
        &
        case_time["case_closed_date"].notna()
        &
        (
            case_time["appeal_date"]
            <
            case_time["case_closed_date"]
        )
    )
)


invalid_case_ids = (
    case_time.loc[
        invalid_case_mask,
        "case_id"
    ]
    .tolist()
)


print(
    "Invalid integrity cases:",
    len(invalid_case_ids)
)


if invalid_case_ids:

    clean_data["integrity_cases"] = (
        clean_data["integrity_cases"][
            ~clean_data[
                "integrity_cases"
            ][
                "case_id"
            ].isin(
                invalid_case_ids
            )
        ]
        .reset_index(drop=True)
    )

    log_cleaning(
        "integrity_cases",
        "*",
        "Impossible case chronology",
        f"Dropped {len(invalid_case_ids)} integrity cases",
        "Case timestamps could not be reliably reconstructed while preserving the investigation timeline."
    )


print(
    "Integrity cases remaining:",
    len(
        clean_data[
            "integrity_cases"
        ]
    )
)

Invalid integrity cases: 11
Integrity cases remaining: 1363


In [63]:
# ============================================================
# DOWNSTREAM DATETIME FINAL CHECK
# ============================================================

# -------------------------
# Detector
# -------------------------

detector_check = (
    clean_data["ai_detector_results"][
        [
            "submission_id",
            "test_timestamp"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "submitted_at"
            ]
        ],
        on="submission_id",
        how="left",
        validate="many_to_one"
    )
)

detector_errors = (
    detector_check["test_timestamp"]
    <=
    detector_check["submitted_at"]
).sum()


# -------------------------
# Behavior events
# -------------------------

behavior_check = (
    clean_data[
        "student_behavior_events"
    ][
        [
            "related_submission_id",
            "student_id",
            "event_date"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        left_on=[
            "related_submission_id",
            "student_id"
        ],
        right_on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)

behavior_errors = (
    behavior_check["event_date"]
    <
    behavior_check["submitted_at"]
).sum()


# -------------------------
# Integrity cases
# -------------------------

case_check = (
    clean_data["integrity_cases"][
        [
            "submission_id",
            "student_id",
            "incident_date",
            "case_open_date",
            "case_closed_date",
            "appeal_date"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id",
                "submitted_at"
            ]
        ],
        on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)

case_errors = {

    "incident_before_submission": int(
        (
            case_check["incident_date"]
            <
            case_check["submitted_at"]
        ).sum()
    ),

    "case_open_before_incident": int(
        (
            case_check["case_open_date"]
            <
            case_check["incident_date"]
        ).sum()
    ),

    "case_closed_before_open": int(
        (
            case_check["case_closed_date"].notna()
            &
            (
                case_check["case_closed_date"]
                <
                case_check["case_open_date"]
            )
        ).sum()
    ),

    "appeal_before_closure": int(
        (
            case_check["appeal_date"].notna()
            &
            case_check["case_closed_date"].notna()
            &
            (
                case_check["appeal_date"]
                <
                case_check["case_closed_date"]
            )
        ).sum()
    )
}


print(
    "Detector timestamp errors:",
    int(detector_errors)
)

print(
    "Behavior timestamp errors:",
    int(behavior_errors)
)

print(
    "\nIntegrity case errors:"
)

for issue, count in case_errors.items():
    print(
        f"{issue}: {count}"
    )

Detector timestamp errors: 0
Behavior timestamp errors: 0

Integrity case errors:
incident_before_submission: 0
case_open_before_incident: 0
case_closed_before_open: 0
appeal_before_closure: 0


In [64]:
# ============================================================
# NUMERIC RANGE RULES
# ============================================================

NUMERIC_RANGE_RULES = {

    "students": {

        "gpa": (0, 4),

        "attendance_rate": (0, 100),

        "expected_graduation_year": (
            2025,
            2035
        )
    },


    "instructors": {

        "years_experience": (
            0,
            50
        ),

        "teaching_load": (
            0,
            10
        )
    },


    "courses": {

        "credits": (
            1,
            6
        )
    },


    "course_offerings": {

        "class_size": (
            1,
            300
        )
    },


    "assignments": {

        "expected_word_count": (
            1,
            np.inf
        ),

        "max_word_count": (
            1,
            np.inf
        ),

        "prompt_complexity": (
            0,
            100
        ),

        "open_endedness": (
            0,
            100
        ),

        "research_intensity": (
            0,
            100
        ),

        "rubric_specificity": (
            0,
            100
        ),

        "required_references": (
            0,
            100
        )
    },


    "submissions": {

        "word_count": (
            1,
            np.inf
        ),

        "cited_references": (
            0,
            100
        ),

        "similarity_score": (
            0,
            1
        ),


        "editing_sessions": (
            0,
            100
        ),

        "revision_count": (
            0,
            100
        ),

        "draft_count": (
            0,
            100
        ),

        "time_spent_minutes": (
            0,
            3000
        ),

        "copy_paste_ratio": (
            0,
            1
        ),

        "number_of_edits": (
            0,
            1000
        ),

        "typing_consistency": (
            0,
            1
        ),

        "avg_sentence_length": (
            1,
            100
        ),

        "sentence_length_std": (
            0,
            50
        ),

        "vocabulary_richness": (
            0,
            1
        ),

        "lexical_diversity": (
            0,
            1
        ),

        "repetition_ratio": (
            0,
            1
        ),

        "grammar_error_rate": (
            0,
            1
        ),

        "citation_density": (
            0,
            100
        ),

        "semantic_coherence": (
            0,
            1
        ),

        "perplexity_score": (
            0,
            1
        ),

        "burstiness_score": (
            0,
            1
        ),

        "grade": (
            0,
            100
        ),

        "feedback_score": (
            0,
            100
        )
    },


    "ai_detector_results": {

        "ai_probability": (
            0,
            1
        ),

        "confidence_score": (
            0,
            1
        ),

        "writing_style_score": (
            0,
            1
        ),

        "detector_perplexity_score": (
            0,
            1
        ),

        "text_length": (
            1,
            50000
        )
    },


    "integrity_cases": {

        "evidence_strength": (
            0,
            100
        ),

        "investigation_duration_days": (
            0,
            365
        ),

        "sanction_points": (
            0,
            100
        )
    },


    "student_integrity_history": {

        "integrity_score": (
            0,
            100
        ),

        "trust_factor_score": (
            0,
            100
        ),

        "prior_confirmed_cases": (
            0,
            np.inf
        ),

        "prior_dismissed_cases": (
            0,
            np.inf
        ),

        "prior_suspicious_flags": (
            0,
            np.inf
        ),

        "prior_ai_flags": (
            0,
            np.inf
        ),

        "prior_plagiarism_flags": (
            0,
            np.inf
        ),

        "prior_sanction_points": (
            0,
            np.inf
        ),

        "recent_integrity_events": (
            0,
            np.inf
        )
    }
}


print(
    "Numeric range rules defined for "
    f"{len(NUMERIC_RANGE_RULES)} tables."
)

Numeric range rules defined for 9 tables.


In [65]:
# ============================================================
# NUMERIC RANGE VIOLATION AUDIT
# ============================================================

numeric_violation_rows = []


for table_name, rules in NUMERIC_RANGE_RULES.items():

    df = clean_data[table_name]

    for column, (
        minimum,
        maximum
    ) in rules.items():

        if column not in df.columns:
            continue

        numeric_series = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        invalid_mask = (
            numeric_series.notna()
            &
            (
                (numeric_series < minimum)
                |
                (numeric_series > maximum)
            )
        )

        invalid_count = int(
            invalid_mask.sum()
        )

        numeric_violation_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "minimum_allowed":
                minimum,

            "maximum_allowed":
                maximum,

            "invalid_count":
                invalid_count
        })


numeric_violation_audit = pd.DataFrame(
    numeric_violation_rows
)


display(
    numeric_violation_audit[
        numeric_violation_audit[
            "invalid_count"
        ] > 0
    ]
)

,table_name,column_name,minimum_allowed,maximum_allowed,invalid_count
0,students,gpa,0,4.0,4
1,students,attendance_rate,0,100.0,3
2,students,expected_graduation_year,2025,2035.0,4
5,courses,credits,1,6.0,1
6,course_offerings,class_size,1,300.0,1
8,assignments,max_word_count,1,inf,1
9,assignments,prompt_complexity,0,100.0,2
10,assignments,open_endedness,0,100.0,2
11,assignments,research_intensity,0,100.0,2
12,assignments,rubric_specificity,0,100.0,2


In [66]:
# ============================================================
# REPAIR INVALID NUMERIC VALUES
# ============================================================

numeric_repair_log = []


for table_name, rules in NUMERIC_RANGE_RULES.items():

    df = clean_data[table_name]

    for column, (
        minimum,
        maximum
    ) in rules.items():

        if column not in df.columns:
            continue

        numeric_series = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        invalid_mask = (
            numeric_series.notna()
            &
            (
                (numeric_series < minimum)
                |
                (numeric_series > maximum)
            )
        )

        invalid_count = int(
            invalid_mask.sum()
        )

        if invalid_count > 0:

            # Invalid values become missing rather than
            # being clipped to an arbitrary boundary.
            df.loc[
                invalid_mask,
                column
            ] = np.nan

            numeric_repair_log.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "invalid_values_replaced":
                    invalid_count
            })

            log_cleaning(
                table_name,
                column,
                "Out-of-range numeric value",
                f"Replaced {invalid_count} invalid values with NaN",
                "The original value cannot be trusted and should not be artificially clipped."
            )


numeric_repair_log = pd.DataFrame(
    numeric_repair_log
)

numeric_repair_log
# ============================================================
# REPAIR INVALID NUMERIC VALUES
# ============================================================

numeric_repair_log = []


for table_name, rules in NUMERIC_RANGE_RULES.items():

    df = clean_data[table_name]

    for column, (
        minimum,
        maximum
    ) in rules.items():

        if column not in df.columns:
            continue

        numeric_series = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        invalid_mask = (
            numeric_series.notna()
            &
            (
                (numeric_series < minimum)
                |
                (numeric_series > maximum)
            )
        )

        invalid_count = int(
            invalid_mask.sum()
        )

        if invalid_count > 0:

            # Invalid values become missing rather than
            # being clipped to an arbitrary boundary.
            df.loc[
                invalid_mask,
                column
            ] = np.nan

            numeric_repair_log.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "invalid_values_replaced":
                    invalid_count
            })

            log_cleaning(
                table_name,
                column,
                "Out-of-range numeric value",
                f"Replaced {invalid_count} invalid values with NaN",
                "The original value cannot be trusted and should not be artificially clipped."
            )


numeric_repair_log = pd.DataFrame(
    numeric_repair_log
)

numeric_repair_log

""


In [67]:
# ============================================================
# FINAL SUBMISSION DERIVED-FIELD RECALCULATION
# ============================================================

submission_recalc = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id",
            "submitted_at"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


submission_recalc[
    "hours_before_deadline"
] = (
    (
        submission_recalc[
            "submission_deadline"
        ]
        -
        submission_recalc[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


submission_recalc[
    "late_submission"
] = (
    submission_recalc[
        "submitted_at"
    ]
    >
    submission_recalc[
        "submission_deadline"
    ]
)


# Update using submission_id
timing_lookup = submission_recalc.set_index(
    "submission_id"
)


clean_data["submissions"][
    "hours_before_deadline"
] = (
    clean_data["submissions"][
        "submission_id"
    ]
    .map(
        timing_lookup[
            "hours_before_deadline"
        ]
    )
    .round(2)
)


clean_data["submissions"][
    "late_submission"
] = (
    clean_data["submissions"][
        "submission_id"
    ]
    .map(
        timing_lookup[
            "late_submission"
        ]
    )
    .astype("boolean")
)


print(
    "Submission-derived fields recalculated."
)

Submission-derived fields recalculated.


In [68]:
# ============================================================
# VALIDATE DERIVED SUBMISSION TIMING
# ============================================================
#
# hours_before_deadline is derived from:
#
# submission_deadline - submitted_at
#
# Therefore it should NOT use a fixed global maximum.
#
# Instead:
#   lower bound = -48 hours
#   upper bound = time available between assignment opening
#                and assignment deadline
# ============================================================


submission_timing_check = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id",
            "submitted_at",
            "hours_before_deadline"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "assignment_open_at",
                "submission_deadline"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# Recalculate authoritative value
# ------------------------------------------------------------

calculated_hours = (
    (
        submission_timing_check[
            "submission_deadline"
        ]
        -
        submission_timing_check[
            "submitted_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


submission_timing_check[
    "calculated_hours"
] = calculated_hours


# ------------------------------------------------------------
# Assignment duration
# ------------------------------------------------------------

assignment_duration = (
    (
        submission_timing_check[
            "submission_deadline"
        ]
        -
        submission_timing_check[
            "assignment_open_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


submission_timing_check[
    "assignment_duration_hours"
] = assignment_duration


# ------------------------------------------------------------
# Check lower bound
# ------------------------------------------------------------

below_lower_bound = (
    submission_timing_check[
        "calculated_hours"
    ] < -48
)


# ------------------------------------------------------------
# Check upper bound
# ------------------------------------------------------------

above_assignment_duration = (
    submission_timing_check[
        "calculated_hours"
    ]
    >
    submission_timing_check[
        "assignment_duration_hours"
    ]
)


# ------------------------------------------------------------
# Check stored value matches calculation
# ------------------------------------------------------------

stored_hours = pd.to_numeric(
    submission_timing_check[
        "hours_before_deadline"
    ],
    errors="coerce"
)


calculation_mismatch = (
    (
        stored_hours
        -
        calculated_hours
    ).abs()
    > 0.05
)


print(
    "Hours below -48:",
    int(below_lower_bound.sum())
)

print(
    "Hours above assignment duration:",
    int(above_assignment_duration.sum())
)

print(
    "Stored/calculated mismatches:",
    int(calculation_mismatch.sum())
)

Hours below -48: 24
Hours above assignment duration: 0
Stored/calculated mismatches: 0


In [69]:
# ============================================================
# MISSING VALUE TREATMENT
# ============================================================
#
# Missing values are handled according to business meaning.
#
# We distinguish between:
# 1. Accidental missing values
# 2. Structurally missing / not-applicable values
# 3. Values that can be safely derived
# 4. Values that should remain missing
# ============================================================


missing_treatment_log = []


def log_missing_treatment(
    table,
    column,
    before,
    after,
    treatment,
    reason
):

    missing_treatment_log.append({

        "table_name": table,

        "column_name": column,

        "missing_before": int(before),

        "missing_after": int(after),

        "values_filled": int(
            before - after
        ),

        "treatment": treatment,

        "reason": reason
    })

In [70]:
# ============================================================
# STUDENT MISSING VALUES
# ============================================================

df = clean_data["students"].copy()


# ------------------------------------------------------------
# GPA
# ------------------------------------------------------------
#
# Use median GPA within major.
# GPA remains bounded between 0 and 4 because invalid values
# were already converted to NaN during numeric cleaning.
# ------------------------------------------------------------

gpa_before = int(
    df["gpa"].isna().sum()
)

major_gpa_median = (
    df.groupby("major_id")["gpa"]
    .transform("median")
)

df["gpa"] = (
    df["gpa"]
    .fillna(major_gpa_median)
)

# Fallback in case an entire major has missing GPA
df["gpa"] = (
    df["gpa"]
    .fillna(
        df["gpa"].median()
    )
)

gpa_after = int(
    df["gpa"].isna().sum()
)

log_missing_treatment(
    "students",
    "gpa",
    gpa_before,
    gpa_after,
    "Median imputation by major",
    "Preserves differences between academic majors without using the ML target."
)


# ------------------------------------------------------------
# Attendance
# ------------------------------------------------------------

attendance_before = int(
    df["attendance_rate"].isna().sum()
)

major_attendance_median = (
    df.groupby("major_id")[
        "attendance_rate"
    ]
    .transform("median")
)

df["attendance_rate"] = (
    df["attendance_rate"]
    .fillna(
        major_attendance_median
    )
)

df["attendance_rate"] = (
    df["attendance_rate"]
    .fillna(
        df["attendance_rate"].median()
    )
)

attendance_after = int(
    df["attendance_rate"].isna().sum()
)

log_missing_treatment(
    "students",
    "attendance_rate",
    attendance_before,
    attendance_after,
    "Median imputation by major",
    "Provides a reasonable value while preserving major-level differences."
)


# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

gender_before = int(
    df["gender"].isna().sum()
)

df["gender"] = (
    df["gender"]
    .fillna("Unknown")
)

gender_after = int(
    df["gender"].isna().sum()
)

log_missing_treatment(
    "students",
    "gender",
    gender_before,
    gender_after,
    "Filled with 'Unknown'",
    "Missing gender does not justify inferring a category."
)


# ------------------------------------------------------------
# Scholarship status
# ------------------------------------------------------------

scholarship_before = int(
    df["scholarship_status"].isna().sum()
)

df["scholarship_status"] = (
    df["scholarship_status"]
    .fillna("Unknown")
)

scholarship_after = int(
    df["scholarship_status"].isna().sum()
)

log_missing_treatment(
    "students",
    "scholarship_status",
    scholarship_before,
    scholarship_after,
    "Filled with 'Unknown'",
    "A missing scholarship value should not be interpreted as no scholarship."
)


# ------------------------------------------------------------
# Academic standing
# ------------------------------------------------------------
#
# This is derived from GPA, so recalculate it instead of
# imputing it independently.
# ------------------------------------------------------------

df["academic_standing"] = np.select(

    [
        df["gpa"] >= 3.70,

        df["gpa"] >= 2.50,

        df["gpa"] >= 2.00
    ],

    [
        "Honors",

        "Good",

        "Academic Warning"
    ],

    default="Probation"
)


academic_missing_after = int(
    df["academic_standing"].isna().sum()
)

print(
    "Remaining missing student values:"
)

print(
    df[
        [
            "gpa",
            "attendance_rate",
            "gender",
            "academic_standing",
            "scholarship_status"
        ]
    ]
    .isna()
    .sum()
)

Remaining missing student values:
gpa                   0
attendance_rate       0
gender                0
academic_standing     0
scholarship_status    0
dtype: int64


In [71]:
# ============================================================
# INSTRUCTOR MISSING VALUES
# ============================================================

df = clean_data["instructors"].copy()


# ------------------------------------------------------------
# Academic rank
# ------------------------------------------------------------

before = int(
    df["academic_rank"].isna().sum()
)

df["academic_rank"] = (
    df["academic_rank"]
    .fillna("Not Recorded")
)

after = int(
    df["academic_rank"].isna().sum()
)

log_missing_treatment(
    "instructors",
    "academic_rank",
    before,
    after,
    "Filled with 'Not Recorded'",
    "Missing faculty rank should not be guessed."
)


# ------------------------------------------------------------
# AI policy adoption
# ------------------------------------------------------------

before = int(
    df["ai_policy_adoption"].isna().sum()
)

df["ai_policy_adoption"] = (
    df["ai_policy_adoption"]
    .fillna("Not Recorded")
)

after = int(
    df["ai_policy_adoption"].isna().sum()
)

log_missing_treatment(
    "instructors",
    "ai_policy_adoption",
    before,
    after,
    "Filled with 'Not Recorded'",
    "Policy adoption cannot be inferred safely from unrelated fields."
)


# ------------------------------------------------------------
# Years of experience
# ------------------------------------------------------------

before = int(
    df["years_experience"].isna().sum()
)

df["years_experience"] = (
    df["years_experience"]
    .fillna(
        df["years_experience"].median()
    )
)

after = int(
    df["years_experience"].isna().sum()
)

log_missing_treatment(
    "instructors",
    "years_experience",
    before,
    after,
    "Median imputation",
    "Small amount of missing numeric experience data."
)


clean_data["instructors"] = df

In [72]:
# ============================================================
# COURSE MISSING VALUES
# ============================================================

df = clean_data["courses"].copy()


# ------------------------------------------------------------
# Course name
# ------------------------------------------------------------

before = int(
    df["course_name"].isna().sum()
)

df["course_name"] = (
    df["course_name"]
    .fillna("Unknown Course")
)

after = int(
    df["course_name"].isna().sum()
)

log_missing_treatment(
    "courses",
    "course_name",
    before,
    after,
    "Filled with 'Unknown Course'",
    "Course ID remains intact and the missing descriptive label cannot be safely reconstructed."
)


# ------------------------------------------------------------
# Course level by department
# ------------------------------------------------------------

before = int(
    df["course_level"].isna().sum()
)

department_course_level = (
    df.groupby("department_id")[
        "course_level"
    ]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else "Intermediate"
    )
)

df["course_level"] = (
    df["course_level"]
    .fillna(
        df["department_id"]
        .map(
            department_course_level
        )
    )
)

df["course_level"] = (
    df["course_level"]
    .fillna("Intermediate")
)

after = int(
    df["course_level"].isna().sum()
)

log_missing_treatment(
    "courses",
    "course_level",
    before,
    after,
    "Mode imputation by department",
    "Course level is related to the academic department structure."
)


# ------------------------------------------------------------
# Difficulty by department
# ------------------------------------------------------------

before = int(
    df["difficulty_level"].isna().sum()
)

department_difficulty = (
    df.groupby("department_id")[
        "difficulty_level"
    ]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else "Moderate"
    )
)

df["difficulty_level"] = (
    df["difficulty_level"]
    .fillna(
        df["department_id"]
        .map(
            department_difficulty
        )
    )
)

df["difficulty_level"] = (
    df["difficulty_level"]
    .fillna("Moderate")
)

after = int(
    df["difficulty_level"].isna().sum()
)

log_missing_treatment(
    "courses",
    "difficulty_level",
    before,
    after,
    "Mode imputation by department",
    "Uses the most common difficulty profile within the academic department."
)


clean_data["courses"] = df

In [73]:
# ============================================================
# ASSIGNMENT MISSING VALUES
# ============================================================

df = clean_data["assignments"].copy()


# ------------------------------------------------------------
# Assignment title
# ------------------------------------------------------------

before = int(
    df["assignment_title"].isna().sum()
)

df["assignment_title"] = (
    df["assignment_title"]
    .fillna("Untitled Assignment")
)

after = int(
    df["assignment_title"].isna().sum()
)

log_missing_treatment(
    "assignments",
    "assignment_title",
    before,
    after,
    "Filled with 'Untitled Assignment'",
    "Missing title does not affect relational integrity."
)


# ------------------------------------------------------------
# Difficulty by assignment type
# ------------------------------------------------------------

before = int(
    df["difficulty_level"].isna().sum()
)

type_difficulty = (
    df.groupby("assignment_type")[
        "difficulty_level"
    ]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else "Moderate"
    )
)

df["difficulty_level"] = (
    df["difficulty_level"]
    .fillna(
        df["assignment_type"]
        .map(type_difficulty)
    )
)

df["difficulty_level"] = (
    df["difficulty_level"]
    .fillna("Moderate")
)

after = int(
    df["difficulty_level"].isna().sum()
)

log_missing_treatment(
    "assignments",
    "difficulty_level",
    before,
    after,
    "Mode imputation by assignment type",
    "Preserves typical difficulty patterns for each assignment type."
)


# ------------------------------------------------------------
# Design type
# ------------------------------------------------------------

before = int(
    df["design_type"].isna().sum()
)

type_design = (
    df.groupby("assignment_type")[
        "design_type"
    ]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else "Traditional"
    )
)

df["design_type"] = (
    df["design_type"]
    .fillna(
        df["assignment_type"]
        .map(type_design)
    )
)

df["design_type"] = (
    df["design_type"]
    .fillna("Traditional")
)

after = int(
    df["design_type"].isna().sum()
)

log_missing_treatment(
    "assignments",
    "design_type",
    before,
    after,
    "Mode imputation by assignment type",
    "Preserves typical assignment-design patterns."
)


# ------------------------------------------------------------
# Required references
# ------------------------------------------------------------
#
# This field has a direct business relationship with
# requires_citations.
#
# If citations are not required → 0.
# If citations are required → use median references for the
# same assignment type.
# ------------------------------------------------------------

before = int(
    df["required_references"].isna().sum()
)

type_reference_median = (
    df.groupby("assignment_type")[
        "required_references"
    ]
    .transform("median")
)

citation_required = (
    df["requires_citations"]
    .astype("boolean")
)


df.loc[
    citation_required == False,
    "required_references"
] = 0


df.loc[
    citation_required == True,
    "required_references"
] = (
    df.loc[
        citation_required == True,
        "required_references"
    ]
    .fillna(
        type_reference_median
    )
)


df["required_references"] = (
    df["required_references"]
    .fillna(0)
    .round()
    .astype(int)
)


after = int(
    df["required_references"].isna().sum()
)

log_missing_treatment(
    "assignments",
    "required_references",
    before,
    after,
    "Derived from citation requirement and assignment-type median",
    "Reference requirements have a direct relationship with citation requirements."
)


clean_data["assignments"] = df

In [74]:
# ============================================================
# SUBMISSION MISSING VALUES
# ============================================================
#
# The target variable `is_ai_generated` is deliberately NOT
# used for imputation.
# ============================================================


df = clean_data["submissions"].copy()


# ------------------------------------------------------------
# Attach assignment context temporarily
# ------------------------------------------------------------

assignment_context = (
    clean_data["assignments"][
        [
            "assignment_id",
            "assignment_type",
            "difficulty_level"
        ]
    ]
)

df = (
    df
    .merge(
        assignment_context,
        on="assignment_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_assignment")
    )
)


submission_numeric_columns = [

    "similarity_score",

    "feedback_score",

    "editing_sessions",

    "draft_count",

    "time_spent_minutes",

    "cited_references",

    "lexical_diversity",

    "vocabulary_richness",

    "revision_count",

    "semantic_coherence",

    "typing_consistency"
]


submission_imputation_log = []


for column in submission_numeric_columns:

    if column not in df.columns:
        continue

    before = int(
        df[column].isna().sum()
    )

    if before == 0:
        continue


    # --------------------------------------------------------
    # Use assignment-type median.
    #
    # We deliberately do NOT use:
    #     is_ai_generated
    #
    # because that is the target.
    # --------------------------------------------------------

    grouped_median = (
        df.groupby(
            "assignment_type"
        )[column]
        .transform("median")
    )

    df[column] = (
        df[column]
        .fillna(grouped_median)
    )


    # Global fallback
    df[column] = (
        df[column]
        .fillna(
            df[column].median()
        )
    )

    after = int(
        df[column].isna().sum()
    )

    submission_imputation_log.append({

        "column": column,

        "missing_before":
            before,

        "missing_after":
            after
    })

    log_missing_treatment(
        "submissions",
        column,
        before,
        after,
        "Median imputation by assignment type",
        "Uses assignment context without referencing the ML target."
    )


# ------------------------------------------------------------
# Recalculate derived timing fields
# ------------------------------------------------------------

df["hours_before_deadline"] = (
    (
        df["submission_deadline"]
        if "submission_deadline" in df.columns
        else pd.NaT
    )
)


# We do not have submission_deadline directly here, so merge
# it temporarily from assignments.
deadline_lookup = clean_data[
    "assignments"
][
    [
        "assignment_id",
        "submission_deadline"
    ]
]


df = (
    df
    .drop(
        columns=[
            "submission_deadline"
        ],
        errors="ignore"
    )
    .merge(
        deadline_lookup,
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


df["hours_before_deadline"] = (
    (
        df["submission_deadline"]
        -
        df["submitted_at"]
    )
    .dt.total_seconds()
    / 3600
).round(2)


df["late_submission"] = (
    df["submitted_at"]
    >
    df["submission_deadline"]
).astype("boolean")


# Remove temporary context columns
df.drop(
    columns=[
        "assignment_type",
        "difficulty_level"
    ],
    inplace=True,
    errors="ignore"
)


clean_data["submissions"] = df


print(
    "Submission missing-value treatment completed."
)

print(
    "Remaining missing submission values:"
)

print(
    clean_data["submissions"]
    .isna()
    .sum()
    .loc[
        lambda x: x > 0
    ]
)

Submission missing-value treatment completed.
Remaining missing submission values:
word_count            16
copy_paste_ratio      30
repetition_ratio      11
grammar_error_rate    16
perplexity_score      30
burstiness_score      30
grade                 30
dtype: int64


In [75]:
# ============================================================
# AI DETECTOR RESULT MISSING VALUES
# ============================================================
#
# Missing detector metrics are preserved when they represent
# failed or partial processing.
#
# We do NOT fabricate detector outputs.
# ============================================================


df = clean_data[
    "ai_detector_results"
].copy()


# Standardize detector-specific null logic.

failed_mask = (
    df["processing_status"]
    == "Failed"
)

partial_mask = (
    df["processing_status"]
    == "Partial"
)


# Missing AI probability / decision for failed processing
# remains missing.

print(
    "Failed detector records:",
    int(failed_mask.sum())
)

print(
    "Partial detector records:",
    int(partial_mask.sum())
)

print(
    "\nMissing detector fields by processing status:"
)

print(
    df.groupby(
        "processing_status",
        dropna=False
    )[
        [
            "ai_probability",
            "detected_as_ai",
            "confidence_score",
            "writing_style_score",
            "detector_perplexity_score",
            "text_length"
        ]
    ]
    .apply(
        lambda x: x.isna().sum()
    )
)

Failed detector records: 764
Partial detector records: 2190

Missing detector fields by processing status:
                   ai_probability  detected_as_ai  confidence_score  \
processing_status                                                     
Failed                        764             764               764   
Partial                         3               0                54   
Success                       108               0              1525   

                   writing_style_score  detector_perplexity_score  text_length  
processing_status                                                               
Failed                             764                        764          764  
Partial                            620                        479          257  
Success                           1534                       1525         1551  


In [76]:
# ============================================================
# STUDENT BEHAVIOR EVENT MISSING VALUES
# ============================================================

df = clean_data[
    "student_behavior_events"
].copy()


# ------------------------------------------------------------
# Recover course_id from related submission
# ------------------------------------------------------------

course_lookup = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "offering_id"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "course_id"
            ]
        ],
        on="offering_id",
        how="left",
        validate="many_to_one"
    )
    [
        [
            "submission_id",
            "course_id"
        ]
    ]
)


df = (
    df
    .merge(
        course_lookup,
        left_on="related_submission_id",
        right_on="submission_id",
        how="left",
        suffixes=("", "_derived")
    )
)


before = int(
    df["course_id"].isna().sum()
)

df["course_id"] = (
    df["course_id"]
    .fillna(
        df["course_id_derived"]
    )
)

after = int(
    df["course_id"].isna().sum()
)

log_missing_treatment(
    "student_behavior_events",
    "course_id",
    before,
    after,
    "Recovered from related submission",
    "Course is deterministically linked through submission → assignment → offering → course."
)


# ------------------------------------------------------------
# Description code from event type
# ------------------------------------------------------------

description_map = {

    "AI Detector Flag":
        "AI_DETECTOR_THRESHOLD",

    "Similarity Flag":
        "SIMILARITY_THRESHOLD",

    "Unusual Collaboration":
        "COLLABORATION_PATTERN",

    "Rapid Submission":
        "RAPID_SUBMISSION",

    "Plagiarism Flag":
        "PLAGIARISM_SCREEN",

    "Excessive Copy-Paste":
        "COPY_PASTE_ANOMALY",

    "Repeated Revision Anomaly":
        "LOW_REVISION_ACTIVITY",

    "Peer Report":
        "PEER_REPORT",

    "Unauthorized File Sharing":
        "FILE_SHARING_MONITOR"
}


before = int(
    df["description_code"].isna().sum()
)

df["description_code"] = (
    df["description_code"]
    .fillna(
        df["event_type"]
        .map(description_map)
    )
)

after = int(
    df["description_code"].isna().sum()
)

log_missing_treatment(
    "student_behavior_events",
    "description_code",
    before,
    after,
    "Derived from event type",
    "Description code is deterministically associated with the event type."
)


# ------------------------------------------------------------
# Severity
# ------------------------------------------------------------

before = int(
    df["severity"].isna().sum()
)

df["severity"] = (
    df["severity"]
    .fillna("Unknown")
)

after = int(
    df["severity"].isna().sum()
)

log_missing_treatment(
    "student_behavior_events",
    "severity",
    before,
    after,
    "Filled with 'Unknown'",
    "Severity should not be invented from unrelated variables."
)


# Remove merge helper columns
df.drop(
    columns=[
        "submission_id",
        "course_id_derived"
    ],
    inplace=True,
    errors="ignore"
)


clean_data[
    "student_behavior_events"
] = df

In [77]:
# ============================================================
# INTEGRITY CASE MISSING VALUES
# ============================================================

df = clean_data[
    "integrity_cases"
].copy()


# ------------------------------------------------------------
# Trigger source
# ------------------------------------------------------------

before = int(
    df["trigger_source"].isna().sum()
)

df["trigger_source"] = (
    df["trigger_source"]
    .fillna("Unknown")
)

after = int(
    df["trigger_source"].isna().sum()
)

log_missing_treatment(
    "integrity_cases",
    "trigger_source",
    before,
    after,
    "Filled with 'Unknown'",
    "The source of a case cannot be safely inferred."
)


# ------------------------------------------------------------
# Student response
# ------------------------------------------------------------

before = int(
    df["student_response"].isna().sum()
)

df["student_response"] = (
    df["student_response"]
    .fillna("No Response")
)

after = int(
    df["student_response"].isna().sum()
)

log_missing_treatment(
    "integrity_cases",
    "student_response",
    before,
    after,
    "Filled with 'No Response'",
    "A missing response is treated as no recorded response rather than inferring an admission or denial."
)


# ------------------------------------------------------------
# Sanction fields
# ------------------------------------------------------------
#
# Keep structural missingness where no sanction applies.
# ------------------------------------------------------------

non_guilty_mask = (
    df["verdict"] != "Guilty"
)

df.loc[
    non_guilty_mask
    &
    df["sanction_level"].isna(),
    "sanction_level"
] = "None"


# Sanction type is genuinely not applicable for
# cases without sanctions, so keep it missing.


# ------------------------------------------------------------
# Pending cases
# ------------------------------------------------------------

pending_mask = (
    df["verdict"] == "Pending"
)

df.loc[
    pending_mask,
    "case_closed_date"
] = pd.NaT

df.loc[
    pending_mask,
    "investigation_duration_days"
] = np.nan


clean_data[
    "integrity_cases"
] = df


print(
    "Integrity-case missing-value treatment completed."
)

print(
    clean_data[
        "integrity_cases"
    ]
    .isna()
    .sum()
    .loc[
        lambda x: x > 0
    ]
)

Integrity-case missing-value treatment completed.
evidence_strength                 4
investigation_duration_days     130
appeal_date                    1225
sanction_type                   989
sanction_points                   3
case_closed_date                127
dtype: int64


In [78]:
# ============================================================
# POST-MISSING-VALUE AUDIT
# ============================================================

remaining_missing_rows = []


for table_name, df in clean_data.items():

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        if missing_count > 0:

            remaining_missing_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "missing_count":
                    missing_count,

                "missing_percentage":
                    round(
                        missing_count
                        / len(df)
                        * 100,
                        2
                    )
            })


remaining_missing_audit = pd.DataFrame(
    remaining_missing_rows
)

remaining_missing_audit = (
    remaining_missing_audit
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


remaining_missing_audit

,table_name,column_name,missing_count,missing_percentage
0,integrity_cases,appeal_date,1225,89.88
1,integrity_cases,sanction_type,989,72.56
2,detector_tools,model_version,1,20.00
3,student_behavior_events,event_value,1794,17.97
4,integrity_cases,investigation_duration_days,130,9.54
5,integrity_cases,case_closed_date,127,9.32
6,ai_detector_results,writing_style_score,2918,3.91
7,ai_detector_results,detector_perplexity_score,2768,3.71
8,ai_detector_results,text_length,2572,3.44
9,ai_detector_results,confidence_score,2343,3.14


In [79]:
# ============================================================
# FINALIZE STUDENT MISSING-VALUE TREATMENT
# ============================================================

df = clean_data["students"].copy()


# ------------------------------------------------------------
# GPA
# ------------------------------------------------------------

before = int(
    df["gpa"].isna().sum()
)

major_gpa_median = (
    df.groupby("major_id")["gpa"]
    .transform("median")
)

df["gpa"] = (
    df["gpa"]
    .fillna(major_gpa_median)
    .fillna(df["gpa"].median())
)

after = int(
    df["gpa"].isna().sum()
)

log_missing_treatment(
    "students",
    "gpa",
    before,
    after,
    "Median imputation by major",
    "GPA is a continuous academic measure and major provides useful context."
)


# ------------------------------------------------------------
# Attendance
# ------------------------------------------------------------

before = int(
    df["attendance_rate"].isna().sum()
)

major_attendance_median = (
    df.groupby("major_id")[
        "attendance_rate"
    ]
    .transform("median")
)

df["attendance_rate"] = (
    df["attendance_rate"]
    .fillna(major_attendance_median)
    .fillna(df["attendance_rate"].median())
)

after = int(
    df["attendance_rate"].isna().sum()
)

log_missing_treatment(
    "students",
    "attendance_rate",
    before,
    after,
    "Median imputation by major",
    "Attendance is numeric and can be reasonably estimated from major-level distributions."
)


# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

before = int(
    df["gender"].isna().sum()
)

df["gender"] = (
    df["gender"]
    .fillna("Unknown")
)

after = int(
    df["gender"].isna().sum()
)

log_missing_treatment(
    "students",
    "gender",
    before,
    after,
    "Filled with 'Unknown'",
    "Missing gender should not be inferred."
)


# ------------------------------------------------------------
# Scholarship status
# ------------------------------------------------------------

before = int(
    df["scholarship_status"].isna().sum()
)

df["scholarship_status"] = (
    df["scholarship_status"]
    .fillna("Unknown")
)

after = int(
    df["scholarship_status"].isna().sum()
)

log_missing_treatment(
    "students",
    "scholarship_status",
    before,
    after,
    "Filled with 'Unknown'",
    "Missing scholarship information should not automatically mean no scholarship."
)


# ------------------------------------------------------------
# Academic standing
# ------------------------------------------------------------
#
# Recalculate from cleaned GPA.
# ------------------------------------------------------------

before = int(
    df["academic_standing"].isna().sum()
)

df["academic_standing"] = np.select(
    [
        df["gpa"] >= 3.70,
        df["gpa"] >= 2.50,
        df["gpa"] >= 2.00
    ],
    [
        "Honors",
        "Good",
        "Academic Warning"
    ],
    default="Probation"
)

after = int(
    df["academic_standing"].isna().sum()
)

log_missing_treatment(
    "students",
    "academic_standing",
    before,
    after,
    "Recalculated from GPA",
    "Academic standing is derived from academic performance."
)


# ------------------------------------------------------------
# WRITE BACK
# ------------------------------------------------------------

clean_data["students"] = (
    df.reset_index(drop=True)
)


print(
    "Student missing-value treatment finalized."
)

print(
    clean_data["students"][
        [
            "gpa",
            "attendance_rate",
            "gender",
            "academic_standing",
            "scholarship_status"
        ]
    ]
    .isna()
    .sum()
)

Student missing-value treatment finalized.
gpa                   0
attendance_rate       0
gender                0
academic_standing     0
scholarship_status    0
dtype: int64


In [80]:
# ============================================================
# FINALIZE SUBMISSION MISSING VALUES
# ============================================================

df = clean_data["submissions"].copy()


# ------------------------------------------------------------
# Attach assignment context
# ------------------------------------------------------------

assignment_context = (
    clean_data["assignments"][
        [
            "assignment_id",
            "assignment_type",
            "expected_word_count",
            "max_word_count"
        ]
    ]
)


df = (
    df
    .merge(
        assignment_context,
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# WORD COUNT
# ------------------------------------------------------------
#
# This can be estimated directly from the assignment's
# expected word count rather than using an arbitrary median.
# ------------------------------------------------------------

before = int(
    df["word_count"].isna().sum()
)

df["word_count"] = (
    df["word_count"]
    .fillna(
        df["expected_word_count"]
    )
)

df["word_count"] = (
    df["word_count"]
    .clip(
        lower=1,
        upper=df["max_word_count"] * 1.05
    )
    .round()
    .astype("Int64")
)

after = int(
    df["word_count"].isna().sum()
)

log_missing_treatment(
    "submissions",
    "word_count",
    before,
    after,
    "Replaced using assignment expected word count",
    "Assignment context provides a direct reference for expected submission length."
)


# ------------------------------------------------------------
# OTHER CONTENT FEATURES
# ------------------------------------------------------------

submission_context_columns = [
    "copy_paste_ratio",
    "repetition_ratio",
    "grammar_error_rate",
    "perplexity_score",
    "burstiness_score",
    "grade"
]


for column in submission_context_columns:

    before = int(
        df[column].isna().sum()
    )

    if before == 0:
        continue


    grouped_median = (
        df.groupby(
            "assignment_type"
        )[column]
        .transform("median")
    )

    df[column] = (
        df[column]
        .fillna(grouped_median)
        .fillna(df[column].median())
    )


    after = int(
        df[column].isna().sum()
    )

    log_missing_treatment(
        "submissions",
        column,
        before,
        after,
        "Median imputation by assignment type",
        "Uses assignment context without referencing the ML target."
    )


# ------------------------------------------------------------
# RECALCULATE DERIVED TIMING FIELDS
# ------------------------------------------------------------

df["hours_before_deadline"] = (
    (
        df["submission_deadline"]
        -
        df["submitted_at"]
    )
    .dt.total_seconds()
    / 3600
).round(2)


df["late_submission"] = (
    df["submitted_at"]
    >
    df["submission_deadline"]
).astype("boolean")


# ------------------------------------------------------------
# Remove temporary assignment columns
# ------------------------------------------------------------

df.drop(
    columns=[
        "assignment_type",
        "expected_word_count",
        "max_word_count",
        "submission_deadline"
    ],
    inplace=True,
    errors="ignore"
)


clean_data["submissions"] = (
    df.reset_index(drop=True)
)


print(
    "Submission missing-value treatment finalized."
)

print(
    clean_data["submissions"]
    .isna()
    .sum()
    .loc[
        lambda x: x > 0
    ]
)

Submission missing-value treatment finalized.
Series([], dtype: int64)


In [81]:
# ============================================================
# HANDLE MISSING CORE DETECTOR RESULTS
# ============================================================
#
# A successful detector analysis should have:
#     ai_probability
#     detected_as_ai
#
# Failed results legitimately lack these values.
#
# We therefore:
# - keep failed records
# - keep partial records with missing auxiliary fields
# - drop successful records missing the core AI probability
# ============================================================


df = clean_data[
    "ai_detector_results"
].copy()


invalid_success_mask = (
    (
        df["processing_status"]
        == "Success"
    )
    &
    (
        df["ai_probability"].isna()
    )
)


invalid_count = int(
    invalid_success_mask.sum()
)


print(
    "Successful detector records missing AI probability:",
    invalid_count
)


if invalid_count > 0:

    invalid_detector_ids = (
        df.loc[
            invalid_success_mask,
            "detector_result_id"
        ]
        .tolist()
    )

    df = (
        df[
            ~df[
                "detector_result_id"
            ].isin(
                invalid_detector_ids
            )
        ]
        .reset_index(drop=True)
    )

    log_cleaning(
        "ai_detector_results",
        "ai_probability",
        "Successful detector result missing core probability",
        f"Dropped {invalid_count} detector result records",
        "A successful detector evaluation without its primary probability cannot be reliably reconstructed."
    )


clean_data[
    "ai_detector_results"
] = df


print(
    "Detector results remaining:",
    len(df)
)

Successful detector records missing AI probability: 108
Detector results remaining: 74561


In [82]:
# ============================================================
# COURSE CREDITS
# ============================================================

df = clean_data["courses"].copy()


before = int(
    df["credits"].isna().sum()
)


subject_credit_mode = (
    df.groupby("subject_area")[
        "credits"
    ]
    .transform(
        lambda x:
            x.mode().iloc[0]
            if not x.mode().empty
            else np.nan
    )
)


df["credits"] = (
    df["credits"]
    .fillna(subject_credit_mode)
    .fillna(df["credits"].median())
    .round()
    .astype("Int64")
)


after = int(
    df["credits"].isna().sum()
)


log_missing_treatment(
    "courses",
    "credits",
    before,
    after,
    "Mode imputation by subject area",
    "Credit structure is related to the academic subject area."
)


clean_data["courses"] = df


print(
    "Remaining missing credits:",
    after
)

Remaining missing credits: 0


In [83]:
# ============================================================
# FINAL MISSING-VALUE REVIEW
# ============================================================

remaining_missing_rows = []


for table_name, df in clean_data.items():

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        if missing_count > 0:

            remaining_missing_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "missing_count":
                    missing_count,

                "missing_percentage":
                    round(
                        missing_count
                        / len(df)
                        * 100,
                        2
                    )
            })


remaining_missing_audit = pd.DataFrame(
    remaining_missing_rows
)

remaining_missing_audit = (
    remaining_missing_audit
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    remaining_missing_audit
)

,table_name,column_name,missing_count,missing_percentage
0,integrity_cases,appeal_date,1225,89.88
1,integrity_cases,sanction_type,989,72.56
2,detector_tools,model_version,1,20.00
3,student_behavior_events,event_value,1794,17.97
4,integrity_cases,investigation_duration_days,130,9.54
5,integrity_cases,case_closed_date,127,9.32
6,ai_detector_results,writing_style_score,2918,3.91
7,ai_detector_results,detector_perplexity_score,2768,3.71
8,ai_detector_results,text_length,2570,3.45
9,ai_detector_results,confidence_score,2342,3.14


In [84]:
# ============================================================
# FOREIGN-KEY VALIDATION
# ============================================================
#
# This stage verifies that every child record references an
# existing parent record after the cleaning process.
#
# No rows are modified in this stage.
# We first measure the remaining relationship problems.
# ============================================================


def count_invalid_fk(
    child_df,
    child_column,
    parent_df,
    parent_column
):
    """
    Count child values that do not exist in the parent table.
    NULL values are excluded because they are handled separately.
    """

    child_values = (
        child_df[child_column]
        .dropna()
        .astype("string")
        .str.strip()
        .str.upper()
    )

    parent_values = (
        parent_df[parent_column]
        .dropna()
        .astype("string")
        .str.strip()
        .str.upper()
        .unique()
    )

    parent_values = set(parent_values)

    return int(
        (~child_values.isin(parent_values)).sum()
    )


foreign_key_rules = [

    (
        "majors",
        "department_id",
        "departments",
        "department_id"
    ),

    (
        "instructors",
        "department_id",
        "departments",
        "department_id"
    ),

    (
        "students",
        "major_id",
        "majors",
        "major_id"
    ),

    (
        "courses",
        "department_id",
        "departments",
        "department_id"
    ),

    (
        "course_offerings",
        "course_id",
        "courses",
        "course_id"
    ),

    (
        "course_offerings",
        "term_id",
        "academic_terms",
        "term_id"
    ),

    (
        "course_offerings",
        "instructor_id",
        "instructors",
        "instructor_id"
    ),

    (
        "assignments",
        "offering_id",
        "course_offerings",
        "offering_id"
    ),

    (
        "submissions",
        "student_id",
        "students",
        "student_id"
    ),

    (
        "submissions",
        "assignment_id",
        "assignments",
        "assignment_id"
    ),

    (
        "ai_detector_results",
        "submission_id",
        "submissions",
        "submission_id"
    ),

    (
        "ai_detector_results",
        "tool_id",
        "detector_tools",
        "tool_id"
    ),

    (
        "student_behavior_events",
        "student_id",
        "students",
        "student_id"
    ),

    (
        "student_behavior_events",
        "course_id",
        "courses",
        "course_id"
    ),

    (
        "student_behavior_events",
        "related_submission_id",
        "submissions",
        "submission_id"
    ),

    (
        "integrity_cases",
        "student_id",
        "students",
        "student_id"
    ),

    (
        "integrity_cases",
        "submission_id",
        "submissions",
        "submission_id"
    ),

    (
        "integrity_cases",
        "assignment_id",
        "assignments",
        "assignment_id"
    ),

    (
        "student_integrity_history",
        "student_id",
        "students",
        "student_id"
    )
]


fk_validation_rows = []


for (
    child_table,
    child_column,
    parent_table,
    parent_column
) in foreign_key_rules:

    child_df = clean_data[
        child_table
    ]

    parent_df = clean_data[
        parent_table
    ]

    null_count = int(
        child_df[
            child_column
        ].isna().sum()
    )

    invalid_count = count_invalid_fk(
        child_df,
        child_column,
        parent_df,
        parent_column
    )

    fk_validation_rows.append({

        "child_table":
            child_table,

        "child_column":
            child_column,

        "parent_table":
            parent_table,

        "parent_column":
            parent_column,

        "child_rows":
            len(child_df),

        "null_fk_values":
            null_count,

        "invalid_fk_values":
            invalid_count
    })


foreign_key_validation = pd.DataFrame(
    fk_validation_rows
)


display(
    foreign_key_validation
)

,child_table,child_column,parent_table,parent_column,child_rows,null_fk_values,invalid_fk_values
0,majors,department_id,departments,department_id,24,0,0
1,instructors,department_id,departments,department_id,100,0,0
2,students,major_id,majors,major_id,1500,0,0
3,courses,department_id,departments,department_id,90,0,0
4,course_offerings,course_id,courses,course_id,261,0,0
5,course_offerings,term_id,academic_terms,term_id,261,0,0
6,course_offerings,instructor_id,instructors,instructor_id,261,0,0
7,assignments,offering_id,course_offerings,offering_id,800,0,0
8,submissions,student_id,students,student_id,14983,0,0
9,submissions,assignment_id,assignments,assignment_id,14983,0,0


In [85]:
# ============================================================
# FOREIGN-KEY ISSUES
# ============================================================

fk_problems = foreign_key_validation[
    (
        foreign_key_validation[
            "invalid_fk_values"
        ] > 0
    )
]


print(
    "Foreign-key relationships with invalid references:",
    len(fk_problems)
)

display(
    fk_problems
)

Foreign-key relationships with invalid references: 0


,child_table,child_column,parent_table,parent_column,child_rows,null_fk_values,invalid_fk_values


In [86]:
# ============================================================
# RELATIONSHIP CARDINALITY VALIDATION
# ============================================================
#
# validate="many_to_one" ensures a child record maps to exactly
# one parent record.
#
# If a parent key is duplicated, pandas raises an exception
# rather than silently multiplying rows.
# ============================================================


relationship_tests = []


# ------------------------------------------------------------
# Student → Major
# ------------------------------------------------------------

student_major_test = (
    clean_data["students"][
        [
            "student_id",
            "major_id"
        ]
    ]
    .merge(
        clean_data["majors"][
            [
                "major_id",
                "department_id"
            ]
        ],
        on="major_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "students → majors",
    "rows_after_join": len(student_major_test),
    "original_rows": len(
        clean_data["students"]
    )
})


# ------------------------------------------------------------
# Course Offering → Course
# ------------------------------------------------------------

offering_course_test = (
    clean_data["course_offerings"][
        [
            "offering_id",
            "course_id"
        ]
    ]
    .merge(
        clean_data["courses"][
            [
                "course_id",
                "department_id"
            ]
        ],
        on="course_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "course_offerings → courses",
    "rows_after_join": len(offering_course_test),
    "original_rows": len(
        clean_data["course_offerings"]
    )
})


# ------------------------------------------------------------
# Offering → Term
# ------------------------------------------------------------

offering_term_test = (
    clean_data["course_offerings"][
        [
            "offering_id",
            "term_id"
        ]
    ]
    .merge(
        clean_data["academic_terms"][
            [
                "term_id",
                "start_date",
                "end_date"
            ]
        ],
        on="term_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "course_offerings → academic_terms",
    "rows_after_join": len(offering_term_test),
    "original_rows": len(
        clean_data["course_offerings"]
    )
})


# ------------------------------------------------------------
# Assignment → Offering
# ------------------------------------------------------------

assignment_offering_test = (
    clean_data["assignments"][
        [
            "assignment_id",
            "offering_id"
        ]
    ]
    .merge(
        clean_data["course_offerings"][
            [
                "offering_id",
                "course_id",
                "term_id"
            ]
        ],
        on="offering_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "assignments → course_offerings",
    "rows_after_join": len(assignment_offering_test),
    "original_rows": len(
        clean_data["assignments"]
    )
})


# ------------------------------------------------------------
# Submission → Assignment
# ------------------------------------------------------------

submission_assignment_test = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "offering_id"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "submissions → assignments",
    "rows_after_join": len(submission_assignment_test),
    "original_rows": len(
        clean_data["submissions"]
    )
})


# ------------------------------------------------------------
# Detector → Submission
# ------------------------------------------------------------

detector_submission_test = (
    clean_data["ai_detector_results"][
        [
            "detector_result_id",
            "submission_id"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id"
            ]
        ],
        on="submission_id",
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "ai_detector_results → submissions",
    "rows_after_join": len(detector_submission_test),
    "original_rows": len(
        clean_data["ai_detector_results"]
    )
})


# ------------------------------------------------------------
# Behavior Event → Submission
# ------------------------------------------------------------

behavior_submission_test = (
    clean_data["student_behavior_events"][
        [
            "event_id",
            "student_id",
            "related_submission_id"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id"
            ]
        ],
        left_on=[
            "related_submission_id",
            "student_id"
        ],
        right_on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "behavior_events → submissions",
    "rows_after_join": len(behavior_submission_test),
    "original_rows": len(
        clean_data["student_behavior_events"]
    )
})


# ------------------------------------------------------------
# Integrity Case → Submission
# ------------------------------------------------------------

case_submission_test = (
    clean_data["integrity_cases"][
        [
            "case_id",
            "student_id",
            "submission_id"
        ]
    ]
    .merge(
        clean_data["submissions"][
            [
                "submission_id",
                "student_id"
            ]
        ],
        on=[
            "submission_id",
            "student_id"
        ],
        how="left",
        validate="many_to_one"
    )
)

relationship_tests.append({
    "relationship": "integrity_cases → submissions",
    "rows_after_join": len(case_submission_test),
    "original_rows": len(
        clean_data["integrity_cases"]
    )
})


relationship_tests = pd.DataFrame(
    relationship_tests
)

display(
    relationship_tests
)

,relationship,rows_after_join,original_rows
0,students → majors,1500,1500
1,course_offerings → courses,261,261
2,course_offerings → academic_terms,261,261
3,assignments → course_offerings,800,800
4,submissions → assignments,14983,14983
5,ai_detector_results → submissions,74561,74561
6,behavior_events → submissions,9984,9984
7,integrity_cases → submissions,1363,1363


In [87]:
# ============================================================
# DETECTOR SUBMISSION-TOOL UNIQUENESS
# ============================================================

detector_pair_duplicates = (
    clean_data["ai_detector_results"][
        [
            "submission_id",
            "tool_id"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate submission-tool pairs:",
    int(
        detector_pair_duplicates
    )
)

Duplicate submission-tool pairs: 0


In [88]:
# ============================================================
# STUDENT HISTORY UNIQUENESS
# ============================================================

history_duplicates = (
    clean_data[
        "student_integrity_history"
    ][
        [
            "student_id",
            "snapshot_at"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate student/snapshot pairs:",
    int(
        history_duplicates
    )
)

Duplicate student/snapshot pairs: 0


In [89]:
# ============================================================
# STUDENT ENROLLMENT / SUBMISSION VALIDATION
# ============================================================

student_submission_context = (
    clean_data["submissions"][
        [
            "submission_id",
            "student_id",
            "assignment_id",
            "submitted_at"
        ]
    ]
    .merge(
        clean_data["students"][
            [
                "student_id",
                "enrollment_date"
            ]
        ],
        on="student_id",
        how="left",
        validate="many_to_one"
    )
)


before_enrollment = (
    student_submission_context[
        "submitted_at"
    ]
    <
    student_submission_context[
        "enrollment_date"
    ]
).sum()


print(
    "Submissions before student enrollment:",
    int(
        before_enrollment
    )
)

Submissions before student enrollment: 0


In [90]:
# ============================================================
# STUDENT GRADUATION-YEAR VALIDATION
# ============================================================

current_year = pd.Timestamp.now().year

students_clean = clean_data[
    "students"
].copy()


graduation_year_errors = (
    (
        students_clean[
            "expected_graduation_year"
        ].notna()
    )
    &
    (
        students_clean[
            "expected_graduation_year"
        ]
        <
        students_clean[
            "enrollment_date"
        ].dt.year
    )
).sum()


print(
    "Graduation year before enrollment year:",
    int(
        graduation_year_errors
    )
)

Graduation year before enrollment year: 0


In [92]:
# ============================================================
# ASSIGNMENT WORD-COUNT VALIDATION
# ============================================================

assignment_word_errors = (
    clean_data["assignments"][
        "max_word_count"
    ]
    <
    clean_data["assignments"][
        "expected_word_count"
    ]
).sum()


print(
    "Assignments with max word count below expected:",
    int(
        assignment_word_errors
    )
)

Assignments with max word count below expected: 2


In [93]:
# ============================================================
# SUBMISSION WORD-COUNT VALIDATION
# ============================================================

submission_word_context = (
    clean_data["submissions"][
        [
            "submission_id",
            "assignment_id",
            "word_count"
        ]
    ]
    .merge(
        clean_data["assignments"][
            [
                "assignment_id",
                "max_word_count"
            ]
        ],
        on="assignment_id",
        how="left",
        validate="many_to_one"
    )
)


submission_word_errors = (
    submission_word_context[
        "word_count"
    ]
    < 1
).sum()


print(
    "Submissions with invalid word count:",
    int(
        submission_word_errors
    )
)

Submissions with invalid word count: 0


In [94]:
# ============================================================
# INTEGRITY CASE BUSINESS RULES
# ============================================================

cases = clean_data[
    "integrity_cases"
].copy()


guilty_without_sanction = (
    (
        cases["verdict"]
        == "Guilty"
    )
    &
    (
        cases["sanction_level"]
        == "None"
    )
).sum()


non_guilty_with_sanction = (
    (
        cases["verdict"]
        != "Guilty"
    )
    &
    (
        cases["sanction_level"]
        .fillna("None")
        != "None"
    )
).sum()


pending_with_closed_case = (
    (
        cases["verdict"]
        == "Pending"
    )
    &
    (
        cases["case_closed_date"]
        .notna()
    )
).sum()


print(
    "Guilty cases without sanction:",
    int(
        guilty_without_sanction
    )
)

print(
    "Non-guilty cases with sanction:",
    int(
        non_guilty_with_sanction
    )
)

print(
    "Pending cases with closure date:",
    int(
        pending_with_closed_case
    )
)

Guilty cases without sanction: 0
Non-guilty cases with sanction: 0
Pending cases with closure date: 0


In [95]:
# ============================================================
# SAVE RELATIONAL AUDIT REPORTS
# ============================================================

foreign_key_validation.to_csv(
    AUDIT_DIR / "11_foreign_key_validation.csv",
    index=False
)

relationship_tests.to_csv(
    AUDIT_DIR / "12_relationship_cardinality.csv",
    index=False
)

print(
    "Relational audit reports saved successfully."
)

Relational audit reports saved successfully.


In [96]:
# ============================================================
# REBUILD STUDENT INTEGRITY HISTORY
# ============================================================
#
# The original history table was generated before the cleaning
# process. It is therefore rebuilt from the finalized:
#
#   students
#   student_behavior_events
#   integrity_cases
#
# Each student receives one snapshot per academic term.
#
# Only cases/events occurring BEFORE the snapshot are included.
# This prevents future-information leakage.
# ============================================================


students_final = clean_data[
    "students"
].copy()

cases_final = clean_data[
    "integrity_cases"
].copy()

events_final = clean_data[
    "student_behavior_events"
].copy()

terms_final = (
    clean_data[
        "academic_terms"
    ]
    .sort_values(
        "start_date"
    )
    .reset_index(
        drop=True
    )
)


history_rows = []

history_counter = 1


for _, student in students_final.iterrows():

    student_id = student[
        "student_id"
    ]

    # --------------------------------------------------------
    # Base trust component
    # --------------------------------------------------------

    gpa_component = (
        (student["gpa"] - 2.5)
        * 7
    )

    attendance_component = (
        (student["attendance_rate"] - 75)
        * 0.12
    )


    for _, term in terms_final.iterrows():

        snapshot_at = (
            pd.Timestamp(
                term["start_date"]
            )
            + pd.Timedelta(
                days=7
            )
        )


        # ====================================================
        # HISTORICAL CASES
        # ====================================================

        historical_cases = cases_final[
            (
                cases_final["student_id"]
                == student_id
            )
            &
            (
                cases_final["case_open_date"]
                <
                snapshot_at
            )
        ]


        prior_confirmed_cases = int(
            (
                historical_cases["verdict"]
                == "Guilty"
            ).sum()
        )


        prior_dismissed_cases = int(
            historical_cases[
                "verdict"
            ].isin(
                [
                    "Dismissed",
                    "Not Guilty",
                    "Insufficient Evidence"
                ]
            ).sum()
        )


        prior_sanction_points = int(
            historical_cases.loc[
                historical_cases[
                    "verdict"
                ] == "Guilty",
                "sanction_points"
            ]
            .fillna(0)
            .sum()
        )


        # ====================================================
        # HISTORICAL BEHAVIOR EVENTS
        # ====================================================

        historical_events = events_final[
            (
                events_final["student_id"]
                == student_id
            )
            &
            (
                events_final["event_date"]
                <
                snapshot_at
            )
        ]


        prior_suspicious_flags = int(
            len(historical_events)
        )


        prior_ai_flags = int(
            (
                historical_events[
                    "event_type"
                ]
                == "AI Detector Flag"
            ).sum()
        )


        prior_plagiarism_flags = int(
            (
                historical_events[
                    "event_type"
                ]
                == "Plagiarism Flag"
            ).sum()
        )


        # ====================================================
        # RECENT EVENTS
        # ====================================================

        recent_start = (
            snapshot_at
            - pd.Timedelta(
                days=90
            )
        )


        recent_integrity_events = int(
            (
                historical_events[
                    "event_date"
                ]
                >=
                recent_start
            ).sum()
        )


        # ====================================================
        # INTEGRITY SCORE
        # ====================================================

        integrity_score = (
            78
            + gpa_component
            + attendance_component

            - (
                prior_confirmed_cases
                * 9
            )

            - (
                prior_dismissed_cases
                * 1.0
            )

            - (
                prior_suspicious_flags
                * 0.65
            )

            - (
                prior_ai_flags
                * 1.0
            )

            - (
                prior_plagiarism_flags
                * 1.5
            )

            - (
                prior_sanction_points
                * 0.50
            )

            - (
                recent_integrity_events
                * 0.70
            )
        )


        # Small recovery for a clean recent history
        if recent_integrity_events == 0:
            integrity_score += 3


        integrity_score = round(
            float(
                np.clip(
                    integrity_score,
                    5,
                    100
                )
            ),
            2
        )


        # ====================================================
        # TRUST FACTOR
        # ====================================================

        trust_factor_score = (
            integrity_score
            + (
                (student["gpa"] - 3.0)
                * 4
            )
            + (
                (student["attendance_rate"] - 85)
                * 0.05
            )
        )


        trust_factor_score = round(
            float(
                np.clip(
                    trust_factor_score,
                    5,
                    100
                )
            ),
            2
        )


        # ====================================================
        # INTEGRITY STATUS
        # ====================================================

        if integrity_score >= 75:

            integrity_status = "High"

        elif integrity_score >= 50:

            integrity_status = "Moderate"

        else:

            integrity_status = "Low"


        # ====================================================
        # BUILD SNAPSHOT
        # ====================================================

        history_rows.append({

            "history_id":
                history_counter,

            "student_id":
                student_id,

            "snapshot_at":
                snapshot_at,

            "integrity_score":
                integrity_score,

            "trust_factor_score":
                trust_factor_score,

            "prior_confirmed_cases":
                prior_confirmed_cases,

            "prior_dismissed_cases":
                prior_dismissed_cases,

            "prior_suspicious_flags":
                prior_suspicious_flags,

            "prior_ai_flags":
                prior_ai_flags,

            "prior_plagiarism_flags":
                prior_plagiarism_flags,

            "prior_sanction_points":
                prior_sanction_points,

            "recent_integrity_events":
                recent_integrity_events,

            "integrity_status":
                integrity_status
        })


        history_counter += 1


student_integrity_history_rebuilt = pd.DataFrame(
    history_rows
)


clean_data[
    "student_integrity_history"
] = (
    student_integrity_history_rebuilt
)


print(
    "Rebuilt history records:",
    len(
        student_integrity_history_rebuilt
    )
)

print(
    "Unique students:",
    student_integrity_history_rebuilt[
        "student_id"
    ].nunique()
)

print(
    "\nSnapshots per student:"
)

print(
    student_integrity_history_rebuilt
    .groupby("student_id")
    .size()
    .value_counts()
    .sort_index()
)

print(
    "\nIntegrity status:"
)

print(
    student_integrity_history_rebuilt[
        "integrity_status"
    ].value_counts()
)

Rebuilt history records: 7500
Unique students: 1500

Snapshots per student:
5    1500
Name: count, dtype: int64

Integrity status:
integrity_status
High        5752
Moderate    1699
Low           49
Name: count, dtype: int64


In [97]:
# ============================================================
# REBUILT HISTORY VALIDATION
# ============================================================

history = clean_data[
    "student_integrity_history"
].copy()


# ------------------------------------------------------------
# Record count
# ------------------------------------------------------------

expected_rows = (
    len(students_final)
    * len(terms_final)
)

print(
    "Expected history records:",
    expected_rows
)

print(
    "Actual history records:",
    len(history)
)


# ------------------------------------------------------------
# Student/snapshot uniqueness
# ------------------------------------------------------------

duplicate_snapshot_keys = (
    history[
        [
            "student_id",
            "snapshot_at"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate student/snapshot pairs:",
    duplicate_snapshot_keys
)


# ------------------------------------------------------------
# Student FK
# ------------------------------------------------------------

invalid_history_students = (
    ~history[
        "student_id"
    ].isin(
        students_final[
            "student_id"
        ]
    )
).sum()

print(
    "Invalid student references:",
    invalid_history_students
)


# ------------------------------------------------------------
# Score ranges
# ------------------------------------------------------------

invalid_integrity_scores = (
    ~history[
        "integrity_score"
    ].between(
        0,
        100
    )
).sum()

invalid_trust_scores = (
    ~history[
        "trust_factor_score"
    ].between(
        0,
        100
    )
).sum()

print(
    "Invalid integrity scores:",
    invalid_integrity_scores
)

print(
    "Invalid trust scores:",
    invalid_trust_scores
)


# ------------------------------------------------------------
# Status consistency
# ------------------------------------------------------------

expected_status = np.select(

    [
        history[
            "integrity_score"
        ] >= 75,

        history[
            "integrity_score"
        ] >= 50
    ],

    [
        "High",
        "Moderate"
    ],

    default="Low"
)


status_errors = (
    history[
        "integrity_status"
    ].astype("string")
    !=
    pd.Series(
        expected_status,
        index=history.index
    )
).sum()

print(
    "Integrity status inconsistencies:",
    status_errors
)

Expected history records: 7500
Actual history records: 7500
Duplicate student/snapshot pairs: 0
Invalid student references: 0
Invalid integrity scores: 0
Invalid trust scores: 0
Integrity status inconsistencies: 0


In [98]:
# ============================================================
# TEMPORAL HISTORY VALIDATION
# ============================================================
#
# For every student/snapshot pair, independently calculate the
# expected historical counts from cases and events occurring
# strictly before the snapshot.
# ============================================================


history_validation = history[
    [
        "student_id",
        "snapshot_at",
        "prior_confirmed_cases",
        "prior_dismissed_cases",
        "prior_suspicious_flags",
        "prior_ai_flags",
        "prior_plagiarism_flags",
        "prior_sanction_points",
        "recent_integrity_events"
    ]
].copy()


validation_rows = []


for _, row in history_validation.iterrows():

    student_id = row[
        "student_id"
    ]

    snapshot = pd.Timestamp(
        row["snapshot_at"]
    )


    student_cases = cases_final[
        (
            cases_final["student_id"]
            == student_id
        )
        &
        (
            cases_final["case_open_date"]
            < snapshot
        )
    ]


    student_events = events_final[
        (
            events_final["student_id"]
            == student_id
        )
        &
        (
            events_final["event_date"]
            < snapshot
        )
    ]


    confirmed = int(
        (
            student_cases["verdict"]
            == "Guilty"
        ).sum()
    )


    dismissed = int(
        student_cases[
            "verdict"
        ].isin(
            [
                "Dismissed",
                "Not Guilty",
                "Insufficient Evidence"
            ]
        ).sum()
    )


    suspicious = int(
        len(student_events)
    )


    ai_flags = int(
        (
            student_events[
                "event_type"
            ]
            == "AI Detector Flag"
        ).sum()
    )


    plagiarism_flags = int(
        (
            student_events[
                "event_type"
            ]
            == "Plagiarism Flag"
        ).sum()
    )


    sanction_points = int(
        student_cases.loc[
            student_cases["verdict"]
            == "Guilty",
            "sanction_points"
        ]
        .fillna(0)
        .sum()
    )


    recent_start = (
        snapshot
        -
        pd.Timedelta(
            days=90
        )
    )


    recent_events = int(
        (
            student_events[
                "event_date"
            ]
            >= recent_start
        ).sum()
    )


    validation_rows.append({

        "student_id":
            student_id,

        "snapshot_at":
            snapshot,

        "expected_confirmed":
            confirmed,

        "actual_confirmed":
            row[
                "prior_confirmed_cases"
            ],

        "expected_dismissed":
            dismissed,

        "actual_dismissed":
            row[
                "prior_dismissed_cases"
            ],

        "expected_suspicious":
            suspicious,

        "actual_suspicious":
            row[
                "prior_suspicious_flags"
            ],

        "expected_ai":
            ai_flags,

        "actual_ai":
            row[
                "prior_ai_flags"
            ],

        "expected_plagiarism":
            plagiarism_flags,

        "actual_plagiarism":
            row[
                "prior_plagiarism_flags"
            ],

        "expected_sanction":
            sanction_points,

        "actual_sanction":
            row[
                "prior_sanction_points"
            ],

        "expected_recent":
            recent_events,

        "actual_recent":
            row[
                "recent_integrity_events"
            ]
    })


history_validation_detail = pd.DataFrame(
    validation_rows
)


comparison_columns = [
    (
        "expected_confirmed",
        "actual_confirmed"
    ),
    (
        "expected_dismissed",
        "actual_dismissed"
    ),
    (
        "expected_suspicious",
        "actual_suspicious"
    ),
    (
        "expected_ai",
        "actual_ai"
    ),
    (
        "expected_plagiarism",
        "actual_plagiarism"
    ),
    (
        "expected_sanction",
        "actual_sanction"
    ),
    (
        "expected_recent",
        "actual_recent"
    )
]


history_error_count = 0


for expected, actual in comparison_columns:

    history_error_count += int(
        (
            history_validation_detail[
                expected
            ]
            !=
            history_validation_detail[
                actual
            ]
        ).sum()
    )


print(
    "Historical count validation errors:",
    history_error_count
)

Historical count validation errors: 0


In [99]:
# ============================================================
# HISTORY TREND SANITY CHECK
# ============================================================

history_trend = (
    history
    .groupby("snapshot_at")
    .agg(
        students=(
            "student_id",
            "nunique"
        ),

        average_integrity_score=(
            "integrity_score",
            "mean"
        ),

        average_trust_score=(
            "trust_factor_score",
            "mean"
        ),

        average_prior_confirmed_cases=(
            "prior_confirmed_cases",
            "mean"
        ),

        average_suspicious_flags=(
            "prior_suspicious_flags",
            "mean"
        )
    )
    .reset_index()
)


history_trend[
    [
        "average_integrity_score",
        "average_trust_score",
        "average_prior_confirmed_cases",
        "average_suspicious_flags"
    ]
] = history_trend[
    [
        "average_integrity_score",
        "average_trust_score",
        "average_prior_confirmed_cases",
        "average_suspicious_flags"
    ]
].round(2)


display(
    history_trend
)

,snapshot_at,students,average_integrity_score,average_trust_score,average_prior_confirmed_cases,average_suspicious_flags
0,2025-08-25,1500,86.23,86.52,0.00,0.00
1,2026-01-19,1500,82.33,82.62,0.05,1.43
2,2026-06-08,1500,79.27,79.56,0.11,2.84
3,2026-08-24,1500,76.22,76.51,0.17,4.14
4,2027-01-18,1500,74.91,75.20,0.21,5.51


In [100]:
# ============================================================
# FINAL MISSING-VALUE REVIEW
# ============================================================

final_missing_rows = []


for table_name, df in clean_data.items():

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        if missing_count > 0:

            final_missing_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "missing_count":
                    missing_count,

                "missing_percentage":
                    round(
                        missing_count
                        / len(df)
                        * 100,
                        2
                    )
            })


final_missing_audit = pd.DataFrame(
    final_missing_rows
)


final_missing_audit = (
    final_missing_audit
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    final_missing_audit
)

,table_name,column_name,missing_count,missing_percentage
0,integrity_cases,appeal_date,1225,89.88
1,integrity_cases,sanction_type,989,72.56
2,detector_tools,model_version,1,20.00
3,student_behavior_events,event_value,1794,17.97
4,integrity_cases,investigation_duration_days,130,9.54
5,integrity_cases,case_closed_date,127,9.32
6,ai_detector_results,writing_style_score,2918,3.91
7,ai_detector_results,detector_perplexity_score,2768,3.71
8,ai_detector_results,text_length,2570,3.45
9,ai_detector_results,confidence_score,2342,3.14


In [101]:
# ============================================================
# FINAL DATA QUALITY SUMMARY
# ============================================================

final_quality_rows = []


for table_name, df in clean_data.items():

    final_quality_rows.append({

        "table_name":
            table_name,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "missing_cells":
            int(
                df.isna()
                .sum()
                .sum()
            ),

        "duplicate_rows":
            int(
                df.duplicated()
                .sum()
            )
    })


final_quality_summary = pd.DataFrame(
    final_quality_rows
)

display(
    final_quality_summary
)

,table_name,rows,columns,missing_cells,duplicate_rows
0,academic_terms,5,6,0,0
1,departments,10,3,0,0
2,majors,24,3,0,0
3,instructors,100,7,0,0
4,students,1500,13,4,0
5,courses,90,8,0,0
6,course_offerings,261,8,11,0
7,assignments,800,20,9,0
8,submissions,14983,32,0,0
9,detector_tools,5,8,1,0


In [102]:
# ============================================================
# FINAL TARGETED MISSING-VALUE CLEANUP
# ============================================================
#
# Only fields where a reasonable value can be inferred from
# existing business context are filled here.
#
# Structurally missing fields remain NULL.
# ============================================================


# ============================================================
# COURSE OFFERINGS
# ============================================================

df = clean_data["course_offerings"].copy()


# ------------------------------------------------------------
# Delivery mode
# ------------------------------------------------------------

before = int(
    df["delivery_mode"].isna().sum()
)

term_delivery_mode = (
    df.groupby("term_id")[
        "delivery_mode"
    ]
    .transform(
        lambda x:
            x.mode().iloc[0]
            if not x.mode().empty
            else "In-Person"
    )
)

df["delivery_mode"] = (
    df["delivery_mode"]
    .fillna(term_delivery_mode)
    .fillna("In-Person")
)

after = int(
    df["delivery_mode"].isna().sum()
)

log_missing_treatment(
    "course_offerings",
    "delivery_mode",
    before,
    after,
    "Mode imputation by term",
    "Delivery mode can reasonably be inferred from the term-level offering pattern."
)


# ------------------------------------------------------------
# Room code
# ------------------------------------------------------------

before = int(
    df["room_code"].isna().sum()
)

df["room_code"] = (
    df["room_code"]
    .fillna("Not Recorded")
)

after = int(
    df["room_code"].isna().sum()
)

log_missing_treatment(
    "course_offerings",
    "room_code",
    before,
    after,
    "Filled with 'Not Recorded'",
    "A missing room cannot be safely inferred."
)


# ------------------------------------------------------------
# Class size
# ------------------------------------------------------------

before = int(
    df["class_size"].isna().sum()
)

course_class_median = (
    df.groupby("course_id")[
        "class_size"
    ]
    .transform("median")
)

df["class_size"] = (
    df["class_size"]
    .fillna(course_class_median)
    .fillna(df["class_size"].median())
    .round()
    .astype("Int64")
)

after = int(
    df["class_size"].isna().sum()
)

log_missing_treatment(
    "course_offerings",
    "class_size",
    before,
    after,
    "Median imputation by course",
    "Class size is associated with the course offering."
)


clean_data["course_offerings"] = (
    df.reset_index(drop=True)
)


# ============================================================
# STUDENTS
# ============================================================

df = clean_data["students"].copy()

before = int(
    df["expected_graduation_year"].isna().sum()
)


major_grad_median = (
    df.groupby("major_id")[
        "expected_graduation_year"
    ]
    .transform("median")
)


df["expected_graduation_year"] = (
    df["expected_graduation_year"]
    .fillna(major_grad_median)
    .fillna(
        df["expected_graduation_year"].median()
    )
    .round()
    .astype("Int64")
)


after = int(
    df["expected_graduation_year"].isna().sum()
)


log_missing_treatment(
    "students",
    "expected_graduation_year",
    before,
    after,
    "Median imputation by major",
    "Graduation timing varies by academic program."
)


clean_data["students"] = (
    df.reset_index(drop=True)
)


# ============================================================
# ASSIGNMENTS
# ============================================================

df = clean_data["assignments"].copy()


assignment_numeric_context = [
    "prompt_complexity",
    "open_endedness",
    "research_intensity",
    "rubric_specificity"
]


for column in assignment_numeric_context:

    before = int(
        df[column].isna().sum()
    )

    if before == 0:
        continue

    grouped_median = (
        df.groupby("assignment_type")[
            column
        ]
        .transform("median")
    )

    df[column] = (
        df[column]
        .fillna(grouped_median)
        .fillna(df[column].median())
    )

    after = int(
        df[column].isna().sum()
    )

    log_missing_treatment(
        "assignments",
        column,
        before,
        after,
        "Median imputation by assignment type",
        "Assignment design metrics are associated with assignment type."
    )


# ------------------------------------------------------------
# Max word count
# ------------------------------------------------------------

before = int(
    df["max_word_count"].isna().sum()
)

df["max_word_count"] = (
    df["max_word_count"]
    .fillna(
        df["expected_word_count"] * 1.20
    )
    .round()
    .astype("Int64")
)

after = int(
    df["max_word_count"].isna().sum()
)

log_missing_treatment(
    "assignments",
    "max_word_count",
    before,
    after,
    "Derived from expected word count",
    "Maximum word count has a direct relationship with expected word count."
)


clean_data["assignments"] = (
    df.reset_index(drop=True)
)


# ============================================================
# INTEGRITY CASES
# ============================================================

df = clean_data["integrity_cases"].copy()


# ------------------------------------------------------------
# Evidence strength
# ------------------------------------------------------------

before = int(
    df["evidence_strength"].isna().sum()
)

case_type_evidence_median = (
    df.groupby("case_type")[
        "evidence_strength"
    ]
    .transform("median")
)

df["evidence_strength"] = (
    df["evidence_strength"]
    .fillna(case_type_evidence_median)
    .fillna(df["evidence_strength"].median())
)

after = int(
    df["evidence_strength"].isna().sum()
)

log_missing_treatment(
    "integrity_cases",
    "evidence_strength",
    before,
    after,
    "Median imputation by case type",
    "Evidence strength is associated with the nature of the case."
)


# ------------------------------------------------------------
# Sanction points
# ------------------------------------------------------------

before = int(
    df["sanction_points"].isna().sum()
)


sanction_point_map = {

    "None": 0,

    "Formal Warning": 1,

    "Assignment Grade Penalty": 4,

    "Course Grade Reduction": 6,

    "Academic Probation": 8,

    "Suspension": 15
}


df["sanction_points"] = (
    df["sanction_points"]
    .fillna(
        df["sanction_type"]
        .map(sanction_point_map)
    )
    .fillna(0)
)


after = int(
    df["sanction_points"].isna().sum()
)


log_missing_treatment(
    "integrity_cases",
    "sanction_points",
    before,
    after,
    "Derived from sanction type",
    "Sanction points are deterministic from the sanction category."
)


clean_data["integrity_cases"] = (
    df.reset_index(drop=True)
)


print(
    "Final targeted missing-value cleanup completed."
)

Final targeted missing-value cleanup completed.


In [103]:
# ============================================================
# FINAL MISSING-VALUE CLASSIFICATION
# ============================================================

final_missing_rows = []


for table_name, df in clean_data.items():

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        if missing_count > 0:

            final_missing_rows.append({

                "table_name":
                    table_name,

                "column_name":
                    column,

                "missing_count":
                    missing_count,

                "missing_percentage":
                    round(
                        missing_count /
                        len(df) *
                        100,
                        2
                    )
            })


final_missing_audit = pd.DataFrame(
    final_missing_rows
)


final_missing_audit = (
    final_missing_audit
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    final_missing_audit
)

,table_name,column_name,missing_count,missing_percentage
0,integrity_cases,appeal_date,1225,89.88
1,integrity_cases,sanction_type,989,72.56
2,detector_tools,model_version,1,20.00
3,student_behavior_events,event_value,1794,17.97
4,integrity_cases,investigation_duration_days,130,9.54
5,integrity_cases,case_closed_date,127,9.32
6,ai_detector_results,writing_style_score,2918,3.91
7,ai_detector_results,detector_perplexity_score,2768,3.71
8,ai_detector_results,text_length,2570,3.45
9,ai_detector_results,confidence_score,2342,3.14


In [104]:
# ============================================================
# GENERATED CLEAN GROUND-TRUTH DATA
# ============================================================
#
# generated_clean contains the original clean synthetic data
# created before intentional corruption was introduced.
#
# It is used only as a reference for evaluating the cleaning
# pipeline.
# ============================================================


ground_truth = {}


for table_name, filename in TABLE_FILES.items():

    file_path = GENERATED_CLEAN_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Ground-truth file not found:\n{file_path}"
        )

    ground_truth[table_name] = pd.read_csv(
        file_path,
        low_memory=False,
        keep_default_na=False,
        na_values=[""]
    )


print(
    f"Ground-truth tables loaded: "
    f"{len(ground_truth)}"
)

for table_name, df in ground_truth.items():

    print(
        f"{table_name:<32}"
        f"{len(df):>9,} rows × "
        f"{len(df.columns):>3} columns"
    )

Ground-truth tables loaded: 14
academic_terms                          5 rows ×   6 columns
departments                            10 rows ×   3 columns
majors                                 24 rows ×   3 columns
instructors                           100 rows ×   7 columns
students                            1,500 rows ×  13 columns
courses                                90 rows ×   8 columns
course_offerings                      261 rows ×   8 columns
assignments                           800 rows ×  20 columns
submissions                        15,000 rows ×  32 columns
detector_tools                          5 rows ×   8 columns
ai_detector_results                75,000 rows ×  11 columns
student_behavior_events            10,037 rows ×  10 columns
integrity_cases                     1,375 rows ×  23 columns
student_integrity_history           7,500 rows ×  13 columns


In [105]:
# ============================================================
# PROCESSED VS GROUND-TRUTH OVERVIEW
# ============================================================

comparison_rows = []


for table_name in TABLE_FILES.keys():

    truth_df = ground_truth[
        table_name
    ]

    processed_df = clean_data[
        table_name
    ]


    truth_rows = len(
        truth_df
    )

    processed_rows = len(
        processed_df
    )

    rows_removed = (
        truth_rows
        -
        processed_rows
    )

    row_recovery = (
        processed_rows
        /
        truth_rows
        * 100
        if truth_rows > 0
        else np.nan
    )


    comparison_rows.append({

        "table_name":
            table_name,

        "ground_truth_rows":
            truth_rows,

        "processed_rows":
            processed_rows,

        "rows_removed":
            rows_removed,

        "row_recovery_percentage":
            round(
                row_recovery,
                2
            ),

        "ground_truth_columns":
            len(
                truth_df.columns
            ),

        "processed_columns":
            len(
                processed_df.columns
            )
    })


processed_vs_truth = pd.DataFrame(
    comparison_rows
)

display(
    processed_vs_truth
)

,table_name,ground_truth_rows,processed_rows,rows_removed,row_recovery_percentage,ground_truth_columns,processed_columns
0,academic_terms,5,5,0,100.00,6,6
1,departments,10,10,0,100.00,3,3
2,majors,24,24,0,100.00,3,3
3,instructors,100,100,0,100.00,7,7
4,students,1500,1500,0,100.00,13,13
5,courses,90,90,0,100.00,8,8
6,course_offerings,261,261,0,100.00,8,8
7,assignments,800,800,0,100.00,20,20
8,submissions,15000,14983,17,99.89,32,32
9,detector_tools,5,5,0,100.00,8,8


In [106]:
# ============================================================
# PRIMARY-KEY RECOVERY COMPARISON
# ============================================================

key_recovery_rows = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    truth_df = ground_truth[
        table_name
    ]

    processed_df = clean_data[
        table_name
    ]


    # --------------------------------------------------------
    # Build normalized composite key strings
    # --------------------------------------------------------

    truth_keys = (
        truth_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )

    processed_keys = (
        processed_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )


    truth_key_set = set(
        truth_keys
    )

    processed_key_set = set(
        processed_keys
    )


    recovered_keys = (
        truth_key_set
        &
        processed_key_set
    )

    removed_keys = (
        truth_key_set
        -
        processed_key_set
    )

    unexpected_keys = (
        processed_key_set
        -
        truth_key_set
    )


    recovery_percentage = (
        len(recovered_keys)
        /
        len(truth_key_set)
        * 100
        if len(truth_key_set) > 0
        else np.nan
    )


    key_recovery_rows.append({

        "table_name":
            table_name,

        "ground_truth_keys":
            len(truth_key_set),

        "processed_keys":
            len(processed_key_set),

        "recovered_keys":
            len(recovered_keys),

        "removed_keys":
            len(removed_keys),

        "unexpected_keys":
            len(unexpected_keys),

        "key_recovery_percentage":
            round(
                recovery_percentage,
                2
            )
    })


key_recovery = pd.DataFrame(
    key_recovery_rows
)

display(
    key_recovery
)

,table_name,ground_truth_keys,processed_keys,recovered_keys,removed_keys,unexpected_keys,key_recovery_percentage
0,academic_terms,5,5,5,0,0,100.00
1,departments,10,10,10,0,0,100.00
2,majors,24,24,24,0,0,100.00
3,instructors,100,100,100,0,0,100.00
4,students,1500,1500,1500,0,0,100.00
5,courses,90,90,90,0,0,100.00
6,course_offerings,261,261,261,0,0,100.00
7,assignments,800,800,800,0,0,100.00
8,submissions,15000,14983,14983,17,0,99.89
9,detector_tools,5,5,5,0,0,100.00


In [107]:
# ============================================================
# REMOVED GROUND-TRUTH KEYS
# ============================================================

removed_key_records = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    truth_df = ground_truth[
        table_name
    ]

    processed_df = clean_data[
        table_name
    ]


    truth_keys = (
        truth_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )

    processed_keys = (
        processed_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )


    processed_key_set = set(
        processed_keys
    )


    removed_mask = (
        ~truth_keys.isin(
            processed_key_set
        )
    )


    if removed_mask.any():

        removed_df = truth_df.loc[
            removed_mask,
            pk_columns
        ].copy()

        removed_df[
            "table_name"
        ] = table_name

        removed_key_records.append(
            removed_df
        )


if removed_key_records:

    removed_ground_truth_keys = pd.concat(
        removed_key_records,
        ignore_index=True
    )

else:

    removed_ground_truth_keys = pd.DataFrame()


print(
    "Ground-truth records removed during cleaning:",
    len(
        removed_ground_truth_keys
    )
)

display(
    removed_ground_truth_keys.head(50)
)

Ground-truth records removed during cleaning: 521


,submission_id,table_name,detector_result_id,event_id,case_id
0,SUB001410,submissions,NaN,NaN,NaN
1,SUB001709,submissions,NaN,NaN,NaN
2,SUB002074,submissions,NaN,NaN,NaN
3,SUB002468,submissions,NaN,NaN,NaN
4,SUB003772,submissions,NaN,NaN,NaN
5,SUB004745,submissions,NaN,NaN,NaN
6,SUB004760,submissions,NaN,NaN,NaN
7,SUB006098,submissions,NaN,NaN,NaN
8,SUB006666,submissions,NaN,NaN,NaN
9,SUB008666,submissions,NaN,NaN,NaN


In [108]:
# ============================================================
# UNEXPECTED PROCESSED KEYS
# ============================================================

unexpected_key_records = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    truth_df = ground_truth[
        table_name
    ]

    processed_df = clean_data[
        table_name
    ]


    truth_keys = (
        truth_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )

    processed_keys = (
        processed_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )


    truth_key_set = set(
        truth_keys
    )


    unexpected_mask = (
        ~processed_keys.isin(
            truth_key_set
        )
    )


    if unexpected_mask.any():

        unexpected_df = processed_df.loc[
            unexpected_mask,
            pk_columns
        ].copy()

        unexpected_df[
            "table_name"
        ] = table_name

        unexpected_key_records.append(
            unexpected_df
        )


if unexpected_key_records:

    unexpected_processed_keys = pd.concat(
        unexpected_key_records,
        ignore_index=True
    )

else:

    unexpected_processed_keys = pd.DataFrame()


print(
    "Unexpected processed keys:",
    len(
        unexpected_processed_keys
    )
)

display(
    unexpected_processed_keys.head(50)
)

Unexpected processed keys: 0


""


In [109]:
# ============================================================
# COLUMN STRUCTURE COMPARISON
# ============================================================

schema_comparison_rows = []


for table_name in TABLE_FILES.keys():

    truth_columns = set(
        ground_truth[
            table_name
        ].columns
    )

    processed_columns = set(
        clean_data[
            table_name
        ].columns
    )


    missing_columns = (
        truth_columns
        -
        processed_columns
    )

    extra_columns = (
        processed_columns
        -
        truth_columns
    )


    schema_comparison_rows.append({

        "table_name":
            table_name,

        "ground_truth_columns":
            len(truth_columns),

        "processed_columns":
            len(processed_columns),

        "missing_columns":
            ", ".join(
                sorted(
                    missing_columns
                )
            ),

        "extra_columns":
            ", ".join(
                sorted(
                    extra_columns
                )
            ),

        "schema_match":
            (
                truth_columns
                ==
                processed_columns
            )
    })


schema_comparison = pd.DataFrame(
    schema_comparison_rows
)

display(
    schema_comparison
)

,table_name,ground_truth_columns,processed_columns,missing_columns,extra_columns,schema_match
0,academic_terms,6,6,,,True
1,departments,3,3,,,True
2,majors,3,3,,,True
3,instructors,7,7,,,True
4,students,13,13,,,True
5,courses,8,8,,,True
6,course_offerings,8,8,,,True
7,assignments,20,20,,,True
8,submissions,32,32,,,True
9,detector_tools,8,8,,,True


In [110]:
# ============================================================
# FIELD-LEVEL VALUE RECOVERY
# ============================================================


def canonicalize_value(
    value,
    column=None
):

    if pd.isna(value):
        return "<NULL>"

    # Normalize datetime values
    if isinstance(
        value,
        (pd.Timestamp, np.datetime64)
    ):

        return str(
            pd.Timestamp(value)
            .round("1s")
        )

    # Normalize strings
    if isinstance(value, str):

        return (
            value
            .strip()
            .casefold()
        )

    # Normalize numeric values
    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        if pd.isna(value):
            return "<NULL>"

        return round(
            float(value),
            6
        )

    return str(value)


field_recovery_rows = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    truth_df = ground_truth[
        table_name
    ].copy()

    processed_df = clean_data[
        table_name
    ].copy()


    # --------------------------------------------------------
    # Index by primary key
    # --------------------------------------------------------

    truth_df["_comparison_key"] = (
        truth_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )

    processed_df["_comparison_key"] = (
        processed_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )


    truth_indexed = (
        truth_df
        .drop_duplicates(
            "_comparison_key"
        )
        .set_index(
            "_comparison_key"
        )
    )

    processed_indexed = (
        processed_df
        .drop_duplicates(
            "_comparison_key"
        )
        .set_index(
            "_comparison_key"
        )
    )


    common_keys = (
        truth_indexed.index
        .intersection(
            processed_indexed.index
        )
    )


    common_columns = [
        col
        for col in truth_df.columns
        if (
            col in processed_df.columns
            and
            col not in ["_comparison_key"]
        )
    ]


    for column in common_columns:

        matches = 0
        comparisons = 0


        for key in common_keys:

            truth_value = canonicalize_value(
                truth_indexed.loc[
                    key,
                    column
                ],
                column
            )

            processed_value = canonicalize_value(
                processed_indexed.loc[
                    key,
                    column
                ],
                column
            )


            comparisons += 1

            if truth_value == processed_value:
                matches += 1


        agreement = (
            matches
            /
            comparisons
            * 100
            if comparisons > 0
            else np.nan
        )


        field_recovery_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "common_records":
                comparisons,

            "matching_values":
                matches,

            "different_values":
                comparisons - matches,

            "agreement_percentage":
                round(
                    agreement,
                    2
                )
        })


field_recovery = pd.DataFrame(
    field_recovery_rows
)

display(
    field_recovery.head(50)
)

,table_name,column_name,common_records,matching_values,different_values,agreement_percentage
0,academic_terms,term_id,5,5,0,100.00
1,academic_terms,term_name,5,5,0,100.00
2,academic_terms,academic_year,5,5,0,100.00
3,academic_terms,term_type,5,5,0,100.00
4,academic_terms,start_date,5,0,5,0.00
5,academic_terms,end_date,5,0,5,0.00
6,departments,department_id,10,10,0,100.00
7,departments,department_name,10,10,0,100.00
8,departments,department_code,10,10,0,100.00
9,majors,major_id,24,24,0,100.00


In [111]:
# ============================================================
# TABLE-LEVEL FIELD RECOVERY
# ============================================================

table_recovery = (
    field_recovery
    .groupby("table_name")
    .agg(
        fields_compared=(
            "column_name",
            "count"
        ),

        average_field_agreement=(
            "agreement_percentage",
            "mean"
        ),

        minimum_field_agreement=(
            "agreement_percentage",
            "min"
        )
    )
    .reset_index()
)


table_recovery[
    [
        "average_field_agreement",
        "minimum_field_agreement"
    ]
] = table_recovery[
    [
        "average_field_agreement",
        "minimum_field_agreement"
    ]
].round(2)


display(
    table_recovery
)

,table_name,fields_compared,average_field_agreement,minimum_field_agreement
0,academic_terms,6,66.67,0.00
1,ai_detector_results,11,81.14,0.00
2,assignments,20,99.62,98.00
3,course_offerings,8,99.57,98.08
4,courses,8,99.86,98.89
5,departments,3,100.00,100.00
6,detector_tools,8,90.00,80.00
7,instructors,7,99.14,98.00
8,integrity_cases,23,86.71,0.00
9,majors,3,100.00,100.00


In [112]:
# ============================================================
# FINAL CLEANING QUALITY SUMMARY
# ============================================================

final_comparison = (
    processed_vs_truth
    .merge(
        key_recovery[
            [
                "table_name",
                "key_recovery_percentage",
                "unexpected_keys"
            ]
        ],
        on="table_name",
        how="left"
    )
    .merge(
        table_recovery[
            [
                "table_name",
                "average_field_agreement"
            ]
        ],
        on="table_name",
        how="left"
    )
)


display(
    final_comparison
)

,table_name,ground_truth_rows,processed_rows,rows_removed,row_recovery_percentage,ground_truth_columns,processed_columns,key_recovery_percentage,unexpected_keys,average_field_agreement
0,academic_terms,5,5,0,100.00,6,6,100.00,0,66.67
1,departments,10,10,0,100.00,3,3,100.00,0,100.00
2,majors,24,24,0,100.00,3,3,100.00,0,100.00
3,instructors,100,100,0,100.00,7,7,100.00,0,99.14
4,students,1500,1500,0,100.00,13,13,100.00,0,91.57
5,courses,90,90,0,100.00,8,8,100.00,0,99.86
6,course_offerings,261,261,0,100.00,8,8,100.00,0,99.57
7,assignments,800,800,0,100.00,20,20,100.00,0,99.62
8,submissions,15000,14983,17,99.89,32,32,99.89,0,96.10
9,detector_tools,5,5,0,100.00,8,8,100.00,0,90.00


In [113]:
# ============================================================
# IMPROVED VALUE CANONICALIZATION
# ============================================================
#
# Ground-truth CSVs are reloaded from disk, so datetime and
# boolean columns may have different pandas dtypes compared
# with the processed in-memory data.
#
# We normalize values based on the column's semantic type
# before comparing them.
# ============================================================


def canonicalize_for_comparison(
    value,
    table_name,
    column_name
):

    if pd.isna(value):
        return "<NULL>"


    # --------------------------------------------------------
    # DATETIME
    # --------------------------------------------------------

    if column_name in DATE_COLUMNS.get(
        table_name,
        []
    ):

        parsed = pd.to_datetime(
            value,
            errors="coerce"
        )

        if pd.isna(parsed):
            return "<INVALID_DATE>"

        return str(
            pd.Timestamp(parsed)
            .round("1s")
        )


    # --------------------------------------------------------
    # BOOLEAN
    # --------------------------------------------------------

    if column_name in BOOLEAN_COLUMNS.get(
        table_name,
        []
    ):

        if isinstance(value, bool):
            return str(value).lower()

        text = (
            str(value)
            .strip()
            .lower()
        )

        boolean_map = {
            "true": "true",
            "false": "false",
            "yes": "true",
            "no": "false",
            "1": "true",
            "0": "false",
            "y": "true",
            "n": "false",
            "t": "true",
            "f": "false"
        }

        return boolean_map.get(
            text,
            text
        )


    # --------------------------------------------------------
    # NUMERIC
    # --------------------------------------------------------

    numeric_rules = NUMERIC_RANGE_RULES.get(
        table_name,
        {}
    )

    if column_name in numeric_rules:

        numeric_value = pd.to_numeric(
            value,
            errors="coerce"
        )

        if pd.notna(numeric_value):

            return round(
                float(numeric_value),
                6
            )


    # --------------------------------------------------------
    # STRING
    # --------------------------------------------------------

    if isinstance(value, str):

        return (
            value
            .strip()
            .casefold()
        )


    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    return str(value)

In [114]:
# ============================================================
# FIELD-LEVEL VALUE RECOVERY - CORRECTED
# ============================================================

field_recovery_rows = []


for table_name, pk_columns in PRIMARY_KEYS.items():

    truth_df = ground_truth[
        table_name
    ].copy()

    processed_df = clean_data[
        table_name
    ].copy()


    # --------------------------------------------------------
    # Build comparison keys
    # --------------------------------------------------------

    truth_df["_comparison_key"] = (
        truth_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )

    processed_df["_comparison_key"] = (
        processed_df[pk_columns]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1
        )
    )


    truth_indexed = (
        truth_df
        .drop_duplicates(
            "_comparison_key"
        )
        .set_index(
            "_comparison_key"
        )
    )

    processed_indexed = (
        processed_df
        .drop_duplicates(
            "_comparison_key"
        )
        .set_index(
            "_comparison_key"
        )
    )


    common_keys = (
        truth_indexed.index
        .intersection(
            processed_indexed.index
        )
    )


    common_columns = [
        column
        for column in truth_df.columns
        if (
            column in processed_df.columns
            and column != "_comparison_key"
        )
    ]


    for column in common_columns:

        matches = 0
        comparisons = 0


        for key in common_keys:

            truth_value = canonicalize_for_comparison(
                truth_indexed.loc[
                    key,
                    column
                ],
                table_name,
                column
            )

            processed_value = canonicalize_for_comparison(
                processed_indexed.loc[
                    key,
                    column
                ],
                table_name,
                column
            )


            comparisons += 1

            if truth_value == processed_value:
                matches += 1


        agreement = (
            matches
            /
            comparisons
            * 100
            if comparisons > 0
            else np.nan
        )


        field_recovery_rows.append({

            "table_name":
                table_name,

            "column_name":
                column,

            "common_records":
                comparisons,

            "matching_values":
                matches,

            "different_values":
                comparisons - matches,

            "agreement_percentage":
                round(
                    agreement,
                    2
                )
        })


field_recovery = pd.DataFrame(
    field_recovery_rows
)


display(
    field_recovery
)

,table_name,column_name,common_records,matching_values,different_values,agreement_percentage
0,academic_terms,term_id,5,5,0,100.00
1,academic_terms,term_name,5,5,0,100.00
2,academic_terms,academic_year,5,5,0,100.00
3,academic_terms,term_type,5,5,0,100.00
4,academic_terms,start_date,5,5,0,100.00
...,...,...,...,...,...,...
160,student_integrity_history,prior_ai_flags,7500,7444,56,99.25
161,student_integrity_history,prior_plagiarism_flags,7500,7486,14,99.81
162,student_integrity_history,prior_sanction_points,7500,7474,26,99.65
163,student_integrity_history,recent_integrity_events,7500,7469,31,99.59


In [115]:
# ============================================================
# CORRECTED TABLE-LEVEL RECOVERY
# ============================================================

table_recovery[
    [
        "average_field_agreement",
        "minimum_field_agreement"
    ]
] = table_recovery[
    [
        "average_field_agreement",
        "minimum_field_agreement"
    ]
].round(2)

display(
    table_recovery
)

,table_name,fields_compared,average_field_agreement,minimum_field_agreement
0,academic_terms,6,66.67,0.00
1,ai_detector_results,11,81.14,0.00
2,assignments,20,99.62,98.00
3,course_offerings,8,99.57,98.08
4,courses,8,99.86,98.89
5,departments,3,100.00,100.00
6,detector_tools,8,90.00,80.00
7,instructors,7,99.14,98.00
8,integrity_cases,23,86.71,0.00
9,majors,3,100.00,100.00


In [116]:
# ============================================================
# FINAL CLEANING COMPARISON
# ============================================================

final_comparison = (
    processed_vs_truth
    .merge(
        key_recovery[
            [
                "table_name",
                "key_recovery_percentage",
                "unexpected_keys"
            ]
        ],
        on="table_name",
        how="left"
    )
    .merge(
        table_recovery[
            [
                "table_name",
                "average_field_agreement"
            ]
        ],
        on="table_name",
        how="left"
    )
)


display(
    final_comparison
)

,table_name,ground_truth_rows,processed_rows,rows_removed,row_recovery_percentage,ground_truth_columns,processed_columns,key_recovery_percentage,unexpected_keys,average_field_agreement
0,academic_terms,5,5,0,100.00,6,6,100.00,0,66.67
1,departments,10,10,0,100.00,3,3,100.00,0,100.00
2,majors,24,24,0,100.00,3,3,100.00,0,100.00
3,instructors,100,100,0,100.00,7,7,100.00,0,99.14
4,students,1500,1500,0,100.00,13,13,100.00,0,91.57
5,courses,90,90,0,100.00,8,8,100.00,0,99.86
6,course_offerings,261,261,0,100.00,8,8,100.00,0,99.57
7,assignments,800,800,0,100.00,20,20,100.00,0,99.62
8,submissions,15000,14983,17,99.89,32,32,99.89,0,96.10
9,detector_tools,5,5,0,100.00,8,8,100.00,0,90.00


In [117]:
# ============================================================
# FINAL CLEAN DATA VALIDATION
# ============================================================

final_validation = []


for table_name, df in clean_data.items():

    final_validation.append({

        "table_name":
            table_name,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "duplicate_rows":
            int(
                df.duplicated().sum()
            ),

        "missing_cells":
            int(
                df.isna().sum().sum()
            )
    })


final_validation = pd.DataFrame(
    final_validation
)


display(
    final_validation
)

,table_name,rows,columns,duplicate_rows,missing_cells
0,academic_terms,5,6,0,0
1,departments,10,3,0,0
2,majors,24,3,0,0
3,instructors,100,7,0,0
4,students,1500,13,0,0
5,courses,90,8,0,0
6,course_offerings,261,8,0,0
7,assignments,800,20,0,0
8,submissions,14983,32,0,0
9,detector_tools,5,8,0,1


In [118]:
# ============================================================
# FINAL CLEANING REPORT
# ============================================================

cleaning_summary = {

    "raw_total_rows": int(
        sum(
            len(df)
            for df in data.values()
        )
    ),

    "processed_total_rows": int(
        sum(
            len(df)
            for df in clean_data.values()
        )
    ),

    "total_rows_removed": int(
        sum(
            len(data[name])
            -
            len(clean_data[name])
            for name in data
        )
    ),

    "tables_processed": len(
        clean_data
    ),

    "tables_with_duplicate_rows": int(
        (
            final_validation[
                "duplicate_rows"
            ] > 0
        ).sum()
    ),

    "tables_with_unexpected_keys": int(
        (
            key_recovery[
                "unexpected_keys"
            ] > 0
        ).sum()
    )
}


print(
    "FINAL EDUSHIELD CLEANING SUMMARY"
)

for key, value in cleaning_summary.items():

    print(
        f"{key}: {value}"
    )

FINAL EDUSHIELD CLEANING SUMMARY
raw_total_rows: 113177
processed_total_rows: 111186
total_rows_removed: 1991
tables_processed: 14
tables_with_duplicate_rows: 0
tables_with_unexpected_keys: 0


In [119]:
# ============================================================
# FINAL EXPORT OF CLEANED DATASETS
# ============================================================
#
# The cleaned datasets are exported in:
#
#   data/processed/
#
# CSV  → easy to inspect / use in Power BI
# Parquet → preserves data types and is efficient for Python/ML
#
# The original raw and generated_clean folders are NOT modified.
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Export inventory
# ------------------------------------------------------------

export_inventory = []


for table_name, df in clean_data.items():

    # --------------------------------------------------------
    # File names
    # --------------------------------------------------------

    csv_path = (
        PROCESSED_DIR
        / f"{table_name}.csv"
    )

    parquet_path = (
        PROCESSED_DIR
        / f"{table_name}.parquet"
    )


    # --------------------------------------------------------
    # CSV export
    # --------------------------------------------------------

    df.to_csv(
        csv_path,
        index=False
    )


    # --------------------------------------------------------
    # Parquet export
    # --------------------------------------------------------

    df.to_parquet(
        parquet_path,
        index=False,
        engine="pyarrow"
    )


    # --------------------------------------------------------
    # Record export details
    # --------------------------------------------------------

    export_inventory.append({

        "table_name":
            table_name,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "csv_file":
            csv_path.name,

        "parquet_file":
            parquet_path.name,

        "missing_cells":
            int(
                df.isna()
                .sum()
                .sum()
            ),

        "duplicate_rows":
            int(
                df.duplicated()
                .sum()
            )
    })


export_inventory = pd.DataFrame(
    export_inventory
)


# ------------------------------------------------------------
# Save export inventory
# ------------------------------------------------------------

export_inventory.to_csv(
    AUDIT_DIR
    / "21_processed_export_inventory.csv",
    index=False
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "=" * 70
)

print(
    "CLEANED DATA EXPORT COMPLETED"
)

print(
    "=" * 70
)

print(
    "\nProcessed data directory:"
)

print(
    PROCESSED_DIR
)

print(
    "\nDatasets exported:",
    len(export_inventory)
)

display(
    export_inventory
)

CLEANED DATA EXPORT COMPLETED

Processed data directory:
c:\Users\adity\Documents\EduShield\data\processed

Datasets exported: 14


,table_name,rows,columns,csv_file,parquet_file,missing_cells,duplicate_rows
0,academic_terms,5,6,academic_terms.csv,academic_terms.parquet,0,0
1,departments,10,3,departments.csv,departments.parquet,0,0
2,majors,24,3,majors.csv,majors.parquet,0,0
3,instructors,100,7,instructors.csv,instructors.parquet,0,0
4,students,1500,13,students.csv,students.parquet,0,0
5,courses,90,8,courses.csv,courses.parquet,0,0
6,course_offerings,261,8,course_offerings.csv,course_offerings.parquet,0,0
7,assignments,800,20,assignments.csv,assignments.parquet,0,0
8,submissions,14983,32,submissions.csv,submissions.parquet,0,0
9,detector_tools,5,8,detector_tools.csv,detector_tools.parquet,1,0
